In [1]:
# ============================================================
# V2: CATBOOST + ROLLING FORM + ELO + SAFE SCORE SELECTION
# Assumption: train, test, sample already loaded
# Current best public score: 3.605218567
# ============================================================

import pandas as pd
import numpy as np
import math
import warnings
warnings.filterwarnings("ignore")

from catboost import CatBoostRegressor


# ============================================================
# 1. BASIC CONFIG
# ============================================================
import pandas as pd
import numpy as np
import glob
import os

# ============================================================
# LOAD DATASET FIRST
# ============================================================

all_csv = glob.glob("/kaggle/input/**/*.csv", recursive=True)

print("CSV files found:")
for f in all_csv:
    print(f)

train_path = None
test_path = None
sample_path = None

for f in all_csv:
    name = os.path.basename(f).lower()
    
    if name == "train.csv":
        train_path = f
    elif name == "test.csv":
        test_path = f
    elif "sample" in name and "submission" in name:
        sample_path = f

print("\nSelected files:")
print("train:", train_path)
print("test:", test_path)
print("sample:", sample_path)

assert train_path is not None, "train.csv not found"
assert test_path is not None, "test.csv not found"
assert sample_path is not None, "sample submission file not found"

train = pd.read_csv(train_path, parse_dates=["date"])
test = pd.read_csv(test_path, parse_dates=["date"])
sample = pd.read_csv(sample_path)

print("\nTrain shape:", train.shape)
print("Test shape:", test.shape)
print("Sample shape:", sample.shape)

display(train.head())
display(test.head())
display(sample.head())


BASE_COLS = list(test.columns)
TARGET_COLS = ["team_goals", "opp_goals"]

CAT_COLS = [
    "gender",
    "team",
    "opponent",
    "tournament",
    "venue_country",
    "confederation_team",
    "confederation_opp",
]

WINDOWS = [3, 5, 10]


# ============================================================
# 2. RESULT + BASE FEATURES
# ============================================================

def add_result_columns_v2(df):
    df = df.copy()

    df["points"] = np.select(
        [
            df["team_goals"] > df["opp_goals"],
            df["team_goals"] == df["opp_goals"],
        ],
        [3, 1],
        default=0,
    )

    df["gd"] = df["team_goals"] - df["opp_goals"]
    df["win"] = (df["team_goals"] > df["opp_goals"]).astype(int)
    df["draw"] = (df["team_goals"] == df["opp_goals"]).astype(int)
    df["loss"] = (df["team_goals"] < df["opp_goals"]).astype(int)

    return df


def add_pair_features(df):
    df = df.copy()

    df.replace(-9999, np.nan, inplace=True)

    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month

    df["same_confederation"] = (
        df["confederation_team"].fillna("Unknown")
        == df["confederation_opp"].fillna("Unknown")
    ).astype(int)

    df["population_diff"] = df["population_team"] - df["population_opp"]
    df["gdp_diff"] = df["gdp_per_capita_team"] - df["gdp_per_capita_opp"]
    df["travel_diff"] = df["distance_travel_team"] - df["distance_travel_opp"]

    df["log_population_team"] = np.log1p(df["population_team"])
    df["log_population_opp"] = np.log1p(df["population_opp"])

    df["home_non_neutral"] = ((df["is_home"] == 1) & (df["neutral"] == 0)).astype(int)
    df["away_non_neutral"] = ((df["is_home"] == 0) & (df["neutral"] == 0)).astype(int)

    tour = df["tournament"].fillna("").str.lower()
    df["is_friendly"] = tour.str.contains("friendly").astype(int)
    df["is_world_cup"] = (
        tour.str.contains("fifa world cup") & ~tour.str.contains("qualification")
    ).astype(int)
    df["is_qualification"] = tour.str.contains("qualification").astype(int)

    return df


# ============================================================
# 3. ROLLING FEATURES FOR TRAINING DATA
# ============================================================

def add_rolling_features_train(history_df):
    df = history_df.copy()
    df.replace(-9999, np.nan, inplace=True)

    df = add_result_columns_v2(df)
    df = df.sort_values(["gender", "team", "date", "match_id"]).copy()

    group = df.groupby(["gender", "team"], sort=False)

    rolling_cols = []

    for w in WINDOWS:
        specs = [
            ("team_goals", f"gf_last{w}", "mean"),
            ("opp_goals", f"ga_last{w}", "mean"),
            ("points", f"points_last{w}", "sum"),
            ("gd", f"gd_last{w}", "sum"),
            ("win", f"win_rate_last{w}", "mean"),
            ("draw", f"draw_rate_last{w}", "mean"),
            ("loss", f"loss_rate_last{w}", "mean"),
        ]

        for source_col, new_col, agg in specs:
            if agg == "mean":
                df[new_col] = group[source_col].transform(
                    lambda s: s.shift(1).rolling(w, min_periods=1).mean()
                )
            else:
                df[new_col] = group[source_col].transform(
                    lambda s: s.shift(1).rolling(w, min_periods=1).sum()
                )

            rolling_cols.append(new_col)

    df["matches_prior"] = group.cumcount()
    prev_date = group["date"].shift(1)
    df["days_since_last_match"] = (df["date"] - prev_date).dt.days

    rolling_cols += ["matches_prior", "days_since_last_match"]

    # Add opponent's rolling features from the paired row in the same match
    opp_map = df[["match_id", "gender", "team"] + rolling_cols].copy()
    opp_map = opp_map.rename(columns={"team": "opponent"})
    opp_map = opp_map.rename(
        columns={c: "opp_" + c for c in rolling_cols}
    )

    df = df.merge(
        opp_map,
        on=["match_id", "gender", "opponent"],
        how="left",
    )

    df = df.sort_index()
    return df


# ============================================================
# 4. ELO FEATURES
# ============================================================

def add_pre_match_elo(history_df, initial=1500.0, k=20.0, home_adv=60.0):
    df = history_df.copy()
    df = df.sort_values(["date", "match_id"]).copy()

    elo = {}
    elo_team = pd.Series(index=df.index, dtype=float)
    elo_opp = pd.Series(index=df.index, dtype=float)

    for match_id, g in df.groupby("match_id", sort=False):
        if len(g) < 2:
            for idx, row in g.iterrows():
                key = (row["gender"], row["team"])
                opp_key = (row["gender"], row["opponent"])
                elo_team.loc[idx] = elo.get(key, initial)
                elo_opp.loc[idx] = elo.get(opp_key, initial)
            continue

        # store pre-match ELO for both perspectives
        for idx, row in g.iterrows():
            key = (row["gender"], row["team"])
            opp_key = (row["gender"], row["opponent"])
            elo_team.loc[idx] = elo.get(key, initial)
            elo_opp.loc[idx] = elo.get(opp_key, initial)

        # update using first row only
        r = g.iloc[0]

        team_key = (r["gender"], r["team"])
        opp_key = (r["gender"], r["opponent"])

        e_team = elo.get(team_key, initial)
        e_opp = elo.get(opp_key, initial)

        if r["team_goals"] > r["opp_goals"]:
            actual = 1.0
        elif r["team_goals"] == r["opp_goals"]:
            actual = 0.5
        else:
            actual = 0.0

        if r["neutral"] == 1:
            adv = 0.0
        else:
            adv = home_adv if r["is_home"] == 1 else -home_adv

        expected = 1.0 / (1.0 + 10.0 ** (-((e_team + adv) - e_opp) / 400.0))
        change = k * (actual - expected)

        elo[team_key] = e_team + change
        elo[opp_key] = e_opp - change

    df["elo_team"] = elo_team
    df["elo_opp"] = elo_opp
    df["elo_diff"] = df["elo_team"] - df["elo_opp"]

    df = df.sort_index()
    return df


def build_final_elo_table(history_df, initial=1500.0, k=20.0, home_adv=60.0):
    df = history_df.copy()
    df = df.sort_values(["date", "match_id"]).copy()

    elo = {}

    first_rows = df.drop_duplicates("match_id", keep="first")

    for _, r in first_rows.iterrows():
        team_key = (r["gender"], r["team"])
        opp_key = (r["gender"], r["opponent"])

        e_team = elo.get(team_key, initial)
        e_opp = elo.get(opp_key, initial)

        if r["team_goals"] > r["opp_goals"]:
            actual = 1.0
        elif r["team_goals"] == r["opp_goals"]:
            actual = 0.5
        else:
            actual = 0.0

        if r["neutral"] == 1:
            adv = 0.0
        else:
            adv = home_adv if r["is_home"] == 1 else -home_adv

        expected = 1.0 / (1.0 + 10.0 ** (-((e_team + adv) - e_opp) / 400.0))
        change = k * (actual - expected)

        elo[team_key] = e_team + change
        elo[opp_key] = e_opp - change

    rows = []
    for (gender, team), value in elo.items():
        rows.append({
            "gender": gender,
            "team": team,
            "elo": value,
        })

    return pd.DataFrame(rows)


# ============================================================
# 5. FINAL STATE FEATURES FOR VALIDATION/TEST
# ============================================================

def build_final_team_state(history_df):
    hist = history_df.copy()
    hist.replace(-9999, np.nan, inplace=True)
    hist = add_result_columns_v2(hist)
    hist = hist.sort_values(["gender", "team", "date", "match_id"])

    rows = []

    for (gender, team), g in hist.groupby(["gender", "team"], sort=False):
        row = {
            "gender": gender,
            "team": team,
            "matches_prior": len(g),
            "last_match_date": g["date"].max(),
        }

        for w in WINDOWS:
            last = g.tail(w)

            row[f"gf_last{w}"] = last["team_goals"].mean()
            row[f"ga_last{w}"] = last["opp_goals"].mean()
            row[f"points_last{w}"] = last["points"].sum()
            row[f"gd_last{w}"] = last["gd"].sum()
            row[f"win_rate_last{w}"] = last["win"].mean()
            row[f"draw_rate_last{w}"] = last["draw"].mean()
            row[f"loss_rate_last{w}"] = last["loss"].mean()

        rows.append(row)

    return pd.DataFrame(rows)


def make_training_features(history_df):
    df = history_df[BASE_COLS + TARGET_COLS].copy()
    df.replace(-9999, np.nan, inplace=True)

    df = add_rolling_features_train(df)
    df = add_pre_match_elo(df)
    df = add_pair_features(df)

    return df


def make_prediction_features_from_history(input_df, history_df):
    df = input_df.copy()
    df.replace(-9999, np.nan, inplace=True)

    state = build_final_team_state(history_df)

    # team state
    df = df.merge(
        state,
        on=["gender", "team"],
        how="left",
    )

    # opponent state
    state_cols = [c for c in state.columns if c not in ["gender", "team"]]
    opp_state = state.rename(columns={"team": "opponent"})
    opp_state = opp_state.rename(
        columns={c: "opp_" + c for c in state_cols}
    )

    df = df.merge(
        opp_state,
        on=["gender", "opponent"],
        how="left",
    )

    # days since last match
    df["days_since_last_match"] = (df["date"] - df["last_match_date"]).dt.days
    df["opp_days_since_last_match"] = (df["date"] - df["opp_last_match_date"]).dt.days

    # ELO
    elo_table = build_final_elo_table(history_df)

    df = df.merge(
        elo_table.rename(columns={"elo": "elo_team"}),
        on=["gender", "team"],
        how="left",
    )

    df = df.merge(
        elo_table.rename(columns={"team": "opponent", "elo": "elo_opp"}),
        on=["gender", "opponent"],
        how="left",
    )

    df["elo_team"] = df["elo_team"].fillna(1500.0)
    df["elo_opp"] = df["elo_opp"].fillna(1500.0)
    df["elo_diff"] = df["elo_team"] - df["elo_opp"]

    df = add_pair_features(df)

    return df


# ============================================================
# 6. MODEL MATRIX
# ============================================================

def get_feature_columns(df):
    exclude = set([
        "Id",
        "match_id",
        "date",
        "team_goals",
        "opp_goals",
        "points",
        "gd",
        "win",
        "draw",
        "loss",
        "last_match_date",
        "opp_last_match_date",
    ])

    cols = [c for c in df.columns if c not in exclude]

    # remove columns that are fully datetime/object unwanted except cat cols
    final_cols = []
    for c in cols:
        if c in CAT_COLS:
            final_cols.append(c)
        elif pd.api.types.is_numeric_dtype(df[c]):
            final_cols.append(c)

    return final_cols


def prepare_X(df, feature_cols, cat_cols, medians=None):
    X = df[feature_cols].copy()

    active_cat_cols = [c for c in cat_cols if c in X.columns]
    num_cols = [c for c in X.columns if c not in active_cat_cols]

    for c in active_cat_cols:
        X[c] = X[c].fillna("Unknown").astype(str)

    if medians is None:
        medians = {}

        for c in num_cols:
            med = X[c].median()
            if pd.isna(med):
                med = 0
            medians[c] = med
            X[c] = X[c].fillna(med)
    else:
        for c in num_cols:
            med = medians.get(c, 0)
            X[c] = X[c].fillna(med)

    return X, medians


# ============================================================
# 7. SCORE SELECTION
# ============================================================

def poisson_logpmf(k, lam):
    lam = max(float(lam), 1e-6)
    return -lam + k * math.log(lam) - math.lgamma(k + 1)


def make_score_predictions_from_lambdas(feature_df, lambdas, max_goals=5):
    temp = feature_df[["Id", "match_id"]].copy()
    temp["lambda"] = np.clip(lambdas, 0.05, 5.50)

    rows = []

    for match_id, g in temp.groupby("match_id", sort=False):
        if len(g) != 2:
            for _, r in g.iterrows():
                tg = int(np.clip(round(r["lambda"]), 0, max_goals))
                rows.append((r["Id"], tg, 1))
            continue

        r1 = g.iloc[0]
        r2 = g.iloc[1]

        lam1 = float(r1["lambda"])
        lam2 = float(r2["lambda"])

        best_score = None
        best_value = -1e18

        for s1 in range(max_goals + 1):
            for s2 in range(max_goals + 1):
                value = poisson_logpmf(s1, lam1) + poisson_logpmf(s2, lam2)

                # common football score prior
                total_goals = s1 + s2
                goal_diff = abs(s1 - s2)

                if total_goals >= 5:
                    value -= 0.18 * (total_goals - 4)

                if goal_diff >= 4:
                    value -= 0.40 * (goal_diff - 3)

                # slight draw support when predicted strengths are close
                if s1 == s2 and abs(lam1 - lam2) < 0.25:
                    value += 0.06

                # avoid too many 0-0 if lambdas are not low
                if s1 == 0 and s2 == 0 and (lam1 + lam2) > 2.30:
                    value -= 0.12

                if value > best_value:
                    best_value = value
                    best_score = (s1, s2)

        s1, s2 = best_score

        rows.append((r1["Id"], s1, s2))
        rows.append((r2["Id"], s2, s1))

    pred = pd.DataFrame(rows, columns=["Id", "team_goals", "opp_goals"])
    return pred


# ============================================================
# 8. POST-PROCESSING MODES
# ============================================================

def compress_score_medium(tg, og):
    tg, og = int(tg), int(og)

    if tg == og:
        if tg >= 2:
            return 1, 1
        return tg, og

    if tg > og:
        diff = tg - og

        if diff >= 4:
            return 3, 0
        elif diff == 3:
            return 2, 0
        elif diff == 2:
            if og >= 2:
                return 2, 1
            return 2, 0
        else:
            if tg + og >= 5:
                return 2, 1
            return tg, og

    if og > tg:
        diff = og - tg

        if diff >= 4:
            return 0, 3
        elif diff == 3:
            return 0, 2
        elif diff == 2:
            if tg >= 2:
                return 1, 2
            return 0, 2
        else:
            if tg + og >= 5:
                return 1, 2
            return tg, og


def compress_score_aggressive(tg, og):
    tg, og = int(tg), int(og)

    if tg == og:
        if tg == 0:
            return 0, 0
        return 1, 1

    if tg > og:
        diff = tg - og
        if diff >= 2:
            return 2, 0
        if og > 0:
            return 2, 1
        return 1, 0

    if og > tg:
        diff = og - tg
        if diff >= 2:
            return 0, 2
        if tg > 0:
            return 1, 2
        return 0, 1


def compress_score_ultra_safe(tg, og):
    tg, og = int(tg), int(og)

    if tg == og:
        if tg == 0:
            return 0, 0
        return 1, 1

    if tg > og:
        if tg - og >= 2:
            return 2, 0
        return 1, 0

    if og > tg:
        if og - tg >= 2:
            return 0, 2
        return 0, 1


def apply_postprocess(sub, mode):
    new_sub = sub.copy()

    if mode == "raw":
        return new_sub

    funcs = {
        "medium": compress_score_medium,
        "aggressive": compress_score_aggressive,
        "ultra_safe": compress_score_ultra_safe,
    }

    func = funcs[mode]

    scores = new_sub.apply(
        lambda r: func(r["team_goals"], r["opp_goals"]),
        axis=1,
    )

    new_sub["team_goals"] = [x[0] for x in scores]
    new_sub["opp_goals"] = [x[1] for x in scores]

    return new_sub


# ============================================================
# 9. LOCAL METRIC APPROXIMATION
# ============================================================

def get_outcome(a, b):
    if a > b:
        return 1
    elif a == b:
        return 0
    else:
        return -1


def tournament_weight(tournament):
    t = str(tournament).lower()

    if "fifa world cup" in t and "qualification" not in t:
        return 2.00
    elif "afc" in t and "championship" in t:
        return 1.80
    elif "friendly" in t:
        return 0.96
    else:
        return 1.20


def approx_aw_mae(y_true_team, y_true_opp, y_pred_team, y_pred_opp, tournaments):
    losses = []
    weights = []

    for atg, aog, ptg, pog, tour in zip(
        y_true_team,
        y_true_opp,
        y_pred_team,
        y_pred_opp,
        tournaments,
    ):
        atg = int(atg)
        aog = int(aog)
        ptg = int(ptg)
        pog = int(pog)

        base_mae = (abs(atg - ptg) + abs(aog - pog)) / 2

        exact_correct = (atg == ptg) and (aog == pog)
        outcome_correct = get_outcome(atg, aog) == get_outcome(ptg, pog)
        gd_correct = (atg - aog) == (ptg - pog)

        exact_penalty = 0 if exact_correct else 0.30
        outcome_penalty = 0 if outcome_correct else 0.25
        gd_penalty = 0 if gd_correct else 0.15

        loss = base_mae + exact_penalty + outcome_penalty + gd_penalty

        if not outcome_correct:
            loss *= 1.5

        loss = loss ** 1.25

        w = tournament_weight(tour)

        losses.append(loss)
        weights.append(w)

    return np.average(losses, weights=weights)


def evaluate_prediction(val_full, pred):
    ev = val_full[["Id", "team_goals", "opp_goals", "tournament"]].merge(
        pred,
        on="Id",
        how="left",
        suffixes=("_true", "_pred"),
    )

    return approx_aw_mae(
        ev["team_goals_true"],
        ev["opp_goals_true"],
        ev["team_goals_pred"],
        ev["opp_goals_pred"],
        ev["tournament"],
    )


# ============================================================
# 10. LOCAL VALIDATION
# ============================================================

cutoff_date = pd.Timestamp("2008-01-01")

hist_train = train[train["date"] < cutoff_date].copy()
val_full = train[train["date"] >= cutoff_date].copy()
val_input = val_full[BASE_COLS].copy()

print("Hist train:", hist_train.shape)
print("Validation:", val_full.shape)
print("Validation date range:", val_full["date"].min(), "to", val_full["date"].max())

hist_fe = make_training_features(hist_train)
val_fe = make_prediction_features_from_history(val_input, hist_train[BASE_COLS + TARGET_COLS].copy())

feature_cols = get_feature_columns(hist_fe)
feature_cols = [c for c in feature_cols if c in val_fe.columns]

active_cat_cols = [c for c in CAT_COLS if c in feature_cols]

X_hist, medians = prepare_X(hist_fe, feature_cols, active_cat_cols)
X_val, _ = prepare_X(val_fe, feature_cols, active_cat_cols, medians=medians)

y_hist = hist_fe["team_goals"].clip(0, 10)

cat_idx = [X_hist.columns.get_loc(c) for c in active_cat_cols]

print("Number of features:", len(feature_cols))
print("Categorical features:", active_cat_cols)

model_val = CatBoostRegressor(
    iterations=900,
    learning_rate=0.035,
    depth=6,
    loss_function="RMSE",
    eval_metric="MAE",
    l2_leaf_reg=8,
    random_seed=42,
    od_type="Iter",
    od_wait=80,
    verbose=200,
)

model_val.fit(
    X_hist,
    y_hist,
    cat_features=cat_idx,
    eval_set=(X_val, val_full["team_goals"].clip(0, 10)),
    use_best_model=True,
)

val_lambda = model_val.predict(X_val)
val_raw_pred = make_score_predictions_from_lambdas(val_fe, val_lambda, max_goals=5)

local_scores = {}

for mode in ["raw", "medium", "aggressive", "ultra_safe"]:
    pred_mode = apply_postprocess(val_raw_pred, mode)
    score = evaluate_prediction(val_full, pred_mode)
    local_scores[mode] = score

print("\nV2 LOCAL VALIDATION SCORES:")
for k, v in sorted(local_scores.items(), key=lambda x: x[1]):
    print(k, ":", v)

best_mode = min(local_scores, key=local_scores.get)
print("\nBest local mode:", best_mode)


# ============================================================
# 11. TRAIN FINAL MODEL ON ALL TRAIN, PREDICT TEST
# ============================================================

all_fe = make_training_features(train)
test_fe = make_prediction_features_from_history(test.copy(), train[BASE_COLS + TARGET_COLS].copy())

final_feature_cols = get_feature_columns(all_fe)
final_feature_cols = [c for c in final_feature_cols if c in test_fe.columns]

final_cat_cols = [c for c in CAT_COLS if c in final_feature_cols]

X_all, final_medians = prepare_X(all_fe, final_feature_cols, final_cat_cols)
X_test, _ = prepare_X(test_fe, final_feature_cols, final_cat_cols, medians=final_medians)

y_all = all_fe["team_goals"].clip(0, 10)

final_cat_idx = [X_all.columns.get_loc(c) for c in final_cat_cols]

best_iter = model_val.get_best_iteration()
if best_iter is None or best_iter < 100:
    best_iter = 800

final_iterations = int(min(max(best_iter + 100, 500), 1200))

print("\nFinal training iterations:", final_iterations)

model_final = CatBoostRegressor(
    iterations=final_iterations,
    learning_rate=0.035,
    depth=6,
    loss_function="RMSE",
    eval_metric="MAE",
    l2_leaf_reg=8,
    random_seed=42,
    verbose=200,
)

model_final.fit(
    X_all,
    y_all,
    cat_features=final_cat_idx,
)

test_lambda = model_final.predict(X_test)

v2_raw = make_score_predictions_from_lambdas(test_fe, test_lambda, max_goals=5)

# align with sample
v2_raw = sample[["Id"]].merge(v2_raw, on="Id", how="left")
v2_raw[["team_goals", "opp_goals"]] = (
    v2_raw[["team_goals", "opp_goals"]]
    .fillna(0)
    .astype(int)
)

v2_medium = apply_postprocess(v2_raw, "medium")
v2_aggressive = apply_postprocess(v2_raw, "aggressive")
v2_ultra = apply_postprocess(v2_raw, "ultra_safe")

v2_raw.to_csv("v2_catboost_raw.csv", index=False)
v2_medium.to_csv("v2_catboost_medium.csv", index=False)
v2_aggressive.to_csv("v2_catboost_aggressive.csv", index=False)
v2_ultra.to_csv("v2_catboost_ultra_safe.csv", index=False)

if best_mode == "raw":
    v2_final = v2_raw.copy()
else:
    v2_final = apply_postprocess(v2_raw, best_mode)

final_name = f"v2_catboost_final_{best_mode}.csv"
v2_final.to_csv(final_name, index=False)

print("\nSaved files:")
print("v2_catboost_raw.csv")
print("v2_catboost_medium.csv")
print("v2_catboost_aggressive.csv")
print("v2_catboost_ultra_safe.csv")
print("Final selected:", final_name)

print("\nFinal submission check:")
print(v2_final.shape)
print(v2_final.isna().sum())
print(v2_final.head(20))

print("\nFinal score distribution:")
display(v2_final[["team_goals", "opp_goals"]].value_counts().head(25))

CSV files found:
/kaggle/input/datasets/athayaakbar/datasatsetsatsetttt/sample submission.csv
/kaggle/input/datasets/athayaakbar/datasatsetsatsetttt/train.csv
/kaggle/input/datasets/athayaakbar/datasatsetsatsetttt/test.csv
/kaggle/input/competitions/deescuy/sample submission.csv
/kaggle/input/competitions/deescuy/train.csv
/kaggle/input/competitions/deescuy/test.csv

Selected files:
train: /kaggle/input/competitions/deescuy/train.csv
test: /kaggle/input/competitions/deescuy/test.csv
sample: /kaggle/input/competitions/deescuy/sample submission.csv

Train shape: (78772, 47)
Test shape: (42422, 20)
Sample shape: (42422, 3)


,Id,match_id,date,gender,team,opponent,is_home,neutral,tournament,venue_country,...,confederation_team,confederation_opp,population_team,population_opp,gdp_per_capita_team,gdp_per_capita_opp,altitude_venue,distance_travel_team,distance_travel_opp,temperature_venue
0,M000001_Scotland,M000001,1872-11-30,M,Scotland,England,1,0,Friendly,Scotland,...,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,M000001_England,M000001,1872-11-30,M,England,Scotland,0,0,Friendly,Scotland,...,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,M000002_England,M000002,1873-03-08,M,England,Scotland,1,0,Friendly,England,...,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,M000002_Scotland,M000002,1873-03-08,M,Scotland,England,0,0,Friendly,England,...,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,M000003_Scotland,M000003,1874-03-07,M,Scotland,England,1,0,Friendly,Scotland,...,UEFA,UEFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Id,match_id,date,gender,team,opponent,is_home,neutral,tournament,venue_country,confederation_team,confederation_opp,population_team,population_opp,gdp_per_capita_team,gdp_per_capita_opp,altitude_venue,distance_travel_team,distance_travel_opp,temperature_venue
0,M034984_Seychelles,M034984,2011-08-06,M,Seychelles,Mauritius,1,0,Indian Ocean Island Games,Seychelles,CAF,CAF,92409.0,1283330.0,12189.095160,9197.026972,-9999.0,0.000000,1751.895724,25.790169
1,M034984_Mauritius,M034984,2011-08-06,M,Mauritius,Seychelles,0,0,Indian Ocean Island Games,Seychelles,CAF,CAF,1283330.0,92409.0,9197.026972,12189.095160,-9999.0,1751.895724,0.000000,25.790169
2,M034985_Comoros,M034985,2011-08-06,M,Comoros,Maldives,1,1,Indian Ocean Island Games,Seychelles,CAF,AFC,656024.0,361575.0,1447.945144,7291.465967,-9999.0,1517.010705,5483.117523,25.790169
3,M034985_Maldives,M034985,2011-08-06,M,Maldives,Comoros,0,1,Indian Ocean Island Games,Seychelles,AFC,CAF,361575.0,656024.0,7291.465967,1447.945144,-9999.0,5483.117523,1517.010705,25.790169
4,M034986_Réunion,M034986,2011-08-06,M,Réunion,Madagascar,1,1,Indian Ocean Island Games,Seychelles,Unknown,CAF,NaN,21731053.0,NaN,531.265432,-9999.0,NaN,NaN,25.790169


,Id,team_goals,opp_goals
0,M034984_Seychelles,0,0
1,M034984_Mauritius,0,0
2,M034985_Comoros,0,0
3,M034985_Maldives,0,0
4,M034986_Réunion,0,0


Hist train: (69842, 47)
Validation: (8930, 47)
Validation date range: 2008-01-02 00:00:00 to 2011-08-04 00:00:00
Number of features: 79
Categorical features: ['gender', 'team', 'opponent', 'tournament', 'venue_country', 'confederation_team', 'confederation_opp']
0:	learn: 1.2599295	test: 1.2260770	best: 1.2260770 (0)	total: 142ms	remaining: 2m 7s
200:	learn: 1.0284548	test: 1.0177677	best: 1.0176551 (199)	total: 12.2s	remaining: 42.5s
400:	learn: 1.0129722	test: 1.0150000	best: 1.0147758 (378)	total: 24.1s	remaining: 30s
Stopped by overfitting detector  (80 iterations wait)

bestTest = 1.014653764
bestIteration = 413

Shrink model to first 414 iterations.

V2 LOCAL VALIDATION SCORES:
raw : 2.3883871009251725
medium : 2.410276725491884
aggressive : 2.410276725491884
ultra_safe : 2.4251426296005465

Best local mode: raw

Final training iterations: 513
0:	learn: 1.2536288	total: 80.6ms	remaining: 41.3s
200:	learn: 1.0217594	total: 13.1s	remaining: 20.3s
400:	learn: 1.0064273	total: 25.7s	

team_goals  opp_goals
1           1            9370
            0            6705
0           1            6705
            2            5250
2           0            5250
3           0            3159
0           3            3159
2           1            1102
1           2            1102
0           0             178
1           3             167
3           1             167
2           2              96
1           4               6
4           1               6
Name: count, dtype: int64

In [2]:
import shutil
import pandas as pd

# File kandidat terbaik dari V2
src = "/kaggle/working/v2_catboost_final_raw.csv"

# Nama final yang Kaggle ambil
dst = "/kaggle/working/submission.csv"

# Copy jadi submission.csv
shutil.copy(src, dst)

# Check final file
submission_check = pd.read_csv(dst)

print("Final file saved as:", dst)
print(submission_check.shape)
print(submission_check.head())
print(submission_check.isna().sum())

assert list(submission_check.columns) == ["Id", "team_goals", "opp_goals"]
assert submission_check.shape[0] == sample.shape[0]
assert submission_check["Id"].equals(sample["Id"])
assert submission_check[["team_goals", "opp_goals"]].isna().sum().sum() == 0

print("READY TO SUBMIT: submission.csv")

Final file saved as: /kaggle/working/submission.csv
(42422, 3)
                   Id  team_goals  opp_goals
0  M034984_Seychelles           1          1
1   M034984_Mauritius           1          1
2     M034985_Comoros           1          1
3    M034985_Maldives           1          1
4     M034986_Réunion           1          0
Id            0
team_goals    0
opp_goals     0
dtype: int64
READY TO SUBMIT: submission.csv


In [3]:
# ============================================================
# V3: CATBOOST REGRESSION + OUTCOME CLASSIFIER
# Final output: submission.csv
# Run this AFTER V2 functions are already defined
# ============================================================

import pandas as pd
import numpy as np
import math
import glob
import os
import warnings
warnings.filterwarnings("ignore")

from catboost import CatBoostRegressor, CatBoostClassifier


# ============================================================
# 0. LOAD DATA IF NOT EXISTS
# ============================================================

if "train" not in globals() or "test" not in globals() or "sample" not in globals():
    all_csv = glob.glob("/kaggle/input/**/*.csv", recursive=True)

    train_path = None
    test_path = None
    sample_path = None

    for f in all_csv:
        name = os.path.basename(f).lower()
        if name == "train.csv":
            train_path = f
        elif name == "test.csv":
            test_path = f
        elif "sample" in name and "submission" in name:
            sample_path = f

    assert train_path is not None, "train.csv not found"
    assert test_path is not None, "test.csv not found"
    assert sample_path is not None, "sample submission file not found"

    train = pd.read_csv(train_path, parse_dates=["date"])
    test = pd.read_csv(test_path, parse_dates=["date"])
    sample = pd.read_csv(sample_path)

print("Train:", train.shape)
print("Test:", test.shape)
print("Sample:", sample.shape)


# ============================================================
# 1. CHECK V2 FUNCTIONS
# ============================================================

required_functions = [
    "make_training_features",
    "make_prediction_features_from_history",
    "get_feature_columns",
    "prepare_X",
    "make_score_predictions_from_lambdas",
    "evaluate_prediction",
    "apply_postprocess",
]

missing_functions = [f for f in required_functions if f not in globals()]

if len(missing_functions) > 0:
    raise Exception(
        "V2 functions belum ada di kernel. Run kode V2 dulu. Missing: "
        + str(missing_functions)
    )


BASE_COLS = list(test.columns)
TARGET_COLS = ["team_goals", "opp_goals"]

CAT_COLS = [
    "gender",
    "team",
    "opponent",
    "tournament",
    "venue_country",
    "confederation_team",
    "confederation_opp",
]


# ============================================================
# 2. OUTCOME HELPERS
# outcome label:
# 0 = lose
# 1 = draw
# 2 = win
# ============================================================

def make_outcome_label(team_goals, opp_goals):
    if team_goals > opp_goals:
        return 2
    elif team_goals == opp_goals:
        return 1
    else:
        return 0


def candidate_outcome_label(s_team, s_opp):
    if s_team > s_opp:
        return 2
    elif s_team == s_opp:
        return 1
    else:
        return 0


def poisson_logpmf_v3(k, lam):
    lam = max(float(lam), 1e-6)
    return -lam + k * math.log(lam) - math.lgamma(k + 1)


# ============================================================
# 3. SCORE SELECTION WITH OUTCOME PROBABILITY
# ============================================================

def make_score_predictions_v3(feature_df, lambdas, outcome_probs, alpha=0.75, max_goals=5):
    """
    alpha controls how strongly outcome classifier influences score selection.
    alpha = 0 means same as pure Poisson-style selection.
    Higher alpha means outcome probability matters more.
    """

    temp = feature_df[["Id", "match_id"]].copy()
    temp["lambda"] = np.clip(lambdas, 0.05, 5.50)

    prob_df = pd.DataFrame(
        outcome_probs,
        columns=["p_loss", "p_draw", "p_win"]
    )

    temp = pd.concat([temp.reset_index(drop=True), prob_df.reset_index(drop=True)], axis=1)

    rows = []

    for match_id, g in temp.groupby("match_id", sort=False):
        if len(g) != 2:
            for _, r in g.iterrows():
                tg = int(np.clip(round(r["lambda"]), 0, max_goals))
                rows.append((r["Id"], tg, 1))
            continue

        r1 = g.iloc[0]
        r2 = g.iloc[1]

        lam1 = float(r1["lambda"])
        lam2 = float(r2["lambda"])

        p1 = np.array([r1["p_loss"], r1["p_draw"], r1["p_win"]], dtype=float)
        p2 = np.array([r2["p_loss"], r2["p_draw"], r2["p_win"]], dtype=float)

        p1 = np.clip(p1, 1e-6, 1.0)
        p2 = np.clip(p2, 1e-6, 1.0)

        best_score = None
        best_value = -1e18

        for s1 in range(max_goals + 1):
            for s2 in range(max_goals + 1):

                value = poisson_logpmf_v3(s1, lam1) + poisson_logpmf_v3(s2, lam2)

                o1 = candidate_outcome_label(s1, s2)
                o2 = candidate_outcome_label(s2, s1)

                # Add outcome probability support
                value += alpha * (math.log(p1[o1]) + math.log(p2[o2]))

                total_goals = s1 + s2
                goal_diff = abs(s1 - s2)

                # Prior supaya tidak terlalu banyak skor besar
                if total_goals >= 5:
                    value -= 0.22 * (total_goals - 4)

                if goal_diff >= 4:
                    value -= 0.50 * (goal_diff - 3)

                # Draw support kalau kekuatan cukup dekat
                if s1 == s2 and abs(lam1 - lam2) < 0.25:
                    value += 0.08

                # Hindari 0-0 kalau expected goals lumayan besar
                if s1 == 0 and s2 == 0 and (lam1 + lam2) > 2.25:
                    value -= 0.15

                # 1-1 prior, karena sering muncul di football
                if s1 == 1 and s2 == 1:
                    value += 0.03

                if value > best_value:
                    best_value = value
                    best_score = (s1, s2)

        s1, s2 = best_score

        rows.append((r1["Id"], s1, s2))
        rows.append((r2["Id"], s2, s1))

    return pd.DataFrame(rows, columns=["Id", "team_goals", "opp_goals"])


# ============================================================
# 4. LOCAL VALIDATION
# ============================================================

cutoff_date = pd.Timestamp("2008-01-01")

hist_train = train[train["date"] < cutoff_date].copy()
val_full = train[train["date"] >= cutoff_date].copy()
val_input = val_full[BASE_COLS].copy()

print("\nHist train:", hist_train.shape)
print("Validation:", val_full.shape)
print("Validation date range:", val_full["date"].min(), "to", val_full["date"].max())

hist_fe = make_training_features(hist_train)
val_fe = make_prediction_features_from_history(
    val_input,
    hist_train[BASE_COLS + TARGET_COLS].copy()
)

feature_cols = get_feature_columns(hist_fe)
feature_cols = [c for c in feature_cols if c in val_fe.columns]

active_cat_cols = [c for c in CAT_COLS if c in feature_cols]

X_hist, medians = prepare_X(hist_fe, feature_cols, active_cat_cols)
X_val, _ = prepare_X(val_fe, feature_cols, active_cat_cols, medians=medians)

cat_idx = [X_hist.columns.get_loc(c) for c in active_cat_cols]

y_goals = hist_fe["team_goals"].clip(0, 10)
y_outcome = [
    make_outcome_label(tg, og)
    for tg, og in zip(hist_fe["team_goals"], hist_fe["opp_goals"])
]

print("Number of features:", len(feature_cols))
print("Categorical features:", active_cat_cols)


# ============================================================
# 5. TRAIN VALIDATION MODELS
# ============================================================

goal_model_val = CatBoostRegressor(
    iterations=900,
    learning_rate=0.035,
    depth=6,
    loss_function="RMSE",
    eval_metric="MAE",
    l2_leaf_reg=8,
    random_seed=42,
    od_type="Iter",
    od_wait=80,
    verbose=200,
)

goal_model_val.fit(
    X_hist,
    y_goals,
    cat_features=cat_idx,
    eval_set=(X_val, val_full["team_goals"].clip(0, 10)),
    use_best_model=True,
)

outcome_model_val = CatBoostClassifier(
    iterations=700,
    learning_rate=0.04,
    depth=6,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    l2_leaf_reg=8,
    random_seed=42,
    od_type="Iter",
    od_wait=80,
    verbose=200,
)

val_outcome_true = [
    make_outcome_label(tg, og)
    for tg, og in zip(val_full["team_goals"], val_full["opp_goals"])
]

outcome_model_val.fit(
    X_hist,
    y_outcome,
    cat_features=cat_idx,
    eval_set=(X_val, val_outcome_true),
    use_best_model=True,
)

val_lambda = goal_model_val.predict(X_val)
val_outcome_probs = outcome_model_val.predict_proba(X_val)


# ============================================================
# 6. TUNE ALPHA LOCALLY
# ============================================================

alpha_grid = [0.00, 0.25, 0.50, 0.75, 1.00, 1.25, 1.50]

alpha_scores = {}

for alpha in alpha_grid:
    pred_alpha = make_score_predictions_v3(
        val_fe,
        val_lambda,
        val_outcome_probs,
        alpha=alpha,
        max_goals=5
    )

    score_raw = evaluate_prediction(val_full, pred_alpha)
    score_medium = evaluate_prediction(val_full, apply_postprocess(pred_alpha, "medium"))
    score_aggressive = evaluate_prediction(val_full, apply_postprocess(pred_alpha, "aggressive"))

    alpha_scores[(alpha, "raw")] = score_raw
    alpha_scores[(alpha, "medium")] = score_medium
    alpha_scores[(alpha, "aggressive")] = score_aggressive

print("\nV3 LOCAL VALIDATION SCORES:")
for (alpha, mode), score in sorted(alpha_scores.items(), key=lambda x: x[1]):
    print(f"alpha={alpha}, mode={mode}: {score}")

best_alpha, best_mode = min(alpha_scores, key=alpha_scores.get)
best_local_score = alpha_scores[(best_alpha, best_mode)]

print("\nBest alpha:", best_alpha)
print("Best mode:", best_mode)
print("Best local score:", best_local_score)


# ============================================================
# 7. TRAIN FINAL MODELS ON ALL TRAIN
# ============================================================

all_fe = make_training_features(train)
test_fe = make_prediction_features_from_history(
    test.copy(),
    train[BASE_COLS + TARGET_COLS].copy()
)

final_feature_cols = get_feature_columns(all_fe)
final_feature_cols = [c for c in final_feature_cols if c in test_fe.columns]

final_cat_cols = [c for c in CAT_COLS if c in final_feature_cols]

X_all, final_medians = prepare_X(all_fe, final_feature_cols, final_cat_cols)
X_test, _ = prepare_X(test_fe, final_feature_cols, final_cat_cols, medians=final_medians)

final_cat_idx = [X_all.columns.get_loc(c) for c in final_cat_cols]

y_all_goals = all_fe["team_goals"].clip(0, 10)

y_all_outcome = [
    make_outcome_label(tg, og)
    for tg, og in zip(all_fe["team_goals"], all_fe["opp_goals"])
]

best_iter_goal = goal_model_val.get_best_iteration()
if best_iter_goal is None or best_iter_goal < 100:
    best_iter_goal = 500

final_goal_iterations = int(min(max(best_iter_goal + 100, 500), 1200))

best_iter_outcome = outcome_model_val.get_best_iteration()
if best_iter_outcome is None or best_iter_outcome < 100:
    best_iter_outcome = 500

final_outcome_iterations = int(min(max(best_iter_outcome + 100, 400), 1000))

print("\nFinal goal iterations:", final_goal_iterations)
print("Final outcome iterations:", final_outcome_iterations)

goal_model_final = CatBoostRegressor(
    iterations=final_goal_iterations,
    learning_rate=0.035,
    depth=6,
    loss_function="RMSE",
    eval_metric="MAE",
    l2_leaf_reg=8,
    random_seed=42,
    verbose=200,
)

goal_model_final.fit(
    X_all,
    y_all_goals,
    cat_features=final_cat_idx,
)

outcome_model_final = CatBoostClassifier(
    iterations=final_outcome_iterations,
    learning_rate=0.04,
    depth=6,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    l2_leaf_reg=8,
    random_seed=42,
    verbose=200,
)

outcome_model_final.fit(
    X_all,
    y_all_outcome,
    cat_features=final_cat_idx,
)


# ============================================================
# 8. PREDICT TEST AND SAVE submission.csv ONLY
# ============================================================

test_lambda = goal_model_final.predict(X_test)
test_outcome_probs = outcome_model_final.predict_proba(X_test)

v3_pred = make_score_predictions_v3(
    test_fe,
    test_lambda,
    test_outcome_probs,
    alpha=best_alpha,
    max_goals=5
)

v3_pred = sample[["Id"]].merge(v3_pred, on="Id", how="left")

v3_pred[["team_goals", "opp_goals"]] = (
    v3_pred[["team_goals", "opp_goals"]]
    .fillna(0)
    .astype(int)
)

if best_mode != "raw":
    v3_pred = apply_postprocess(v3_pred, best_mode)

# IMPORTANT: final name only submission.csv
v3_pred.to_csv("submission.csv", index=False)

print("\nSaved final file: submission.csv")
print("Best alpha:", best_alpha)
print("Best mode:", best_mode)
print("Best local score:", best_local_score)

print("\nSubmission check:")
print(v3_pred.shape)
print(v3_pred.isna().sum())
print(v3_pred.head(20))

assert list(v3_pred.columns) == ["Id", "team_goals", "opp_goals"]
assert v3_pred.shape[0] == sample.shape[0]
assert v3_pred["Id"].equals(sample["Id"])
assert v3_pred[["team_goals", "opp_goals"]].isna().sum().sum() == 0

print("\nScore distribution:")
display(v3_pred[["team_goals", "opp_goals"]].value_counts().head(30))

print("\nREADY TO SUBMIT: /kaggle/working/submission.csv")

Train: (78772, 47)
Test: (42422, 20)
Sample: (42422, 3)

Hist train: (69842, 47)
Validation: (8930, 47)
Validation date range: 2008-01-02 00:00:00 to 2011-08-04 00:00:00
Number of features: 79
Categorical features: ['gender', 'team', 'opponent', 'tournament', 'venue_country', 'confederation_team', 'confederation_opp']
0:	learn: 1.2599295	test: 1.2260770	best: 1.2260770 (0)	total: 77.4ms	remaining: 1m 9s
200:	learn: 1.0284548	test: 1.0177677	best: 1.0176551 (199)	total: 12.1s	remaining: 42.2s
400:	learn: 1.0129722	test: 1.0150000	best: 1.0147758 (378)	total: 24s	remaining: 29.9s
Stopped by overfitting detector  (80 iterations wait)

bestTest = 1.014653764
bestIteration = 413

Shrink model to first 414 iterations.
0:	learn: 1.0853138	test: 1.0851091	best: 1.0851091 (0)	total: 249ms	remaining: 2m 54s
200:	learn: 0.8704919	test: 0.8893082	best: 0.8892316 (188)	total: 44s	remaining: 1m 49s
400:	learn: 0.8539774	test: 0.8878987	best: 0.8877413 (387)	total: 1m 27s	remaining: 1m 5s
600:	learn:

team_goals  opp_goals
0           1            8328
1           0            8328
0           2            5250
2           0            5250
1           1            4238
0           3            3159
3           0            3159
1           2            2120
2           1            2120
1           3             169
3           1             169
0           0             116
4           1               5
1           4               5
2           3               3
3           2               3
Name: count, dtype: int64


READY TO SUBMIT: /kaggle/working/submission.csv


versi 4

In [4]:
# ============================================================
# V4: GOALS + OUTCOME + GOAL DIFFERENCE + TOTAL GOALS
# Final output: submission.csv
# Run AFTER V2/V3 functions are already defined
# ============================================================

import pandas as pd
import numpy as np
import math
import glob
import os
import warnings
warnings.filterwarnings("ignore")

from catboost import CatBoostRegressor, CatBoostClassifier


# ============================================================
# 0. LOAD DATA IF NEEDED
# ============================================================

if "train" not in globals() or "test" not in globals() or "sample" not in globals():
    all_csv = glob.glob("/kaggle/input/**/*.csv", recursive=True)

    train_path = None
    test_path = None
    sample_path = None

    for f in all_csv:
        name = os.path.basename(f).lower()
        if name == "train.csv":
            train_path = f
        elif name == "test.csv":
            test_path = f
        elif "sample" in name and "submission" in name:
            sample_path = f

    assert train_path is not None, "train.csv not found"
    assert test_path is not None, "test.csv not found"
    assert sample_path is not None, "sample submission file not found"

    train = pd.read_csv(train_path, parse_dates=["date"])
    test = pd.read_csv(test_path, parse_dates=["date"])
    sample = pd.read_csv(sample_path)

print("Train:", train.shape)
print("Test:", test.shape)
print("Sample:", sample.shape)


# ============================================================
# 1. CHECK REQUIRED FUNCTIONS FROM V2/V3
# ============================================================

required_functions = [
    "make_training_features",
    "make_prediction_features_from_history",
    "get_feature_columns",
    "prepare_X",
    "evaluate_prediction",
    "apply_postprocess",
]

missing_functions = [f for f in required_functions if f not in globals()]

if len(missing_functions) > 0:
    raise Exception(
        "Run kode V2/V3 dulu. Missing functions: " + str(missing_functions)
    )


BASE_COLS = list(test.columns)
TARGET_COLS = ["team_goals", "opp_goals"]

CAT_COLS = [
    "gender",
    "team",
    "opponent",
    "tournament",
    "venue_country",
    "confederation_team",
    "confederation_opp",
]


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def make_outcome_label(team_goals, opp_goals):
    if team_goals > opp_goals:
        return 2      # win
    elif team_goals == opp_goals:
        return 1      # draw
    else:
        return 0      # lose


def candidate_outcome_label(s_team, s_opp):
    if s_team > s_opp:
        return 2
    elif s_team == s_opp:
        return 1
    else:
        return 0


def poisson_logpmf_v4(k, lam):
    lam = max(float(lam), 1e-6)
    return -lam + k * math.log(lam) - math.lgamma(k + 1)


# ============================================================
# 3. V4 SCORE SELECTOR
# ============================================================

def make_score_predictions_v4(
    feature_df,
    lambdas,
    outcome_probs,
    diff_preds,
    total_preds,
    alpha=0.25,
    beta_diff=0.18,
    beta_total=0.10,
    max_goals=5,
    total_penalty=0.22,
    diff_penalty=0.50,
    draw_boost=0.08,
    oneone_boost=0.03,
):
    """
    Select final score per match using:
    - goal lambda
    - outcome probability
    - predicted goal difference
    - predicted total goals
    """

    temp = feature_df[["Id", "match_id"]].copy()
    temp["lambda"] = np.clip(lambdas, 0.05, 5.50)
    temp["pred_diff"] = np.clip(diff_preds, -6, 6)
    temp["pred_total"] = np.clip(total_preds, 0, 8)

    prob_df = pd.DataFrame(
        outcome_probs,
        columns=["p_loss", "p_draw", "p_win"]
    )

    temp = pd.concat([temp.reset_index(drop=True), prob_df.reset_index(drop=True)], axis=1)

    rows = []

    common_scores = {
        (1, 0): 0.05,
        (0, 1): 0.05,
        (1, 1): oneone_boost,
        (2, 0): 0.03,
        (0, 2): 0.03,
        (2, 1): 0.03,
        (1, 2): 0.03,
        (0, 0): 0.02,
    }

    for match_id, g in temp.groupby("match_id", sort=False):

        if len(g) != 2:
            for _, r in g.iterrows():
                tg = int(np.clip(round(r["lambda"]), 0, max_goals))
                rows.append((r["Id"], tg, 1))
            continue

        r1 = g.iloc[0]
        r2 = g.iloc[1]

        lam1 = float(r1["lambda"])
        lam2 = float(r2["lambda"])

        p1 = np.array([r1["p_loss"], r1["p_draw"], r1["p_win"]], dtype=float)
        p2 = np.array([r2["p_loss"], r2["p_draw"], r2["p_win"]], dtype=float)

        p1 = np.clip(p1, 1e-6, 1.0)
        p2 = np.clip(p2, 1e-6, 1.0)

        d1 = float(r1["pred_diff"])
        d2 = float(r2["pred_diff"])

        t1 = float(r1["pred_total"])
        t2 = float(r2["pred_total"])

        best_score = None
        best_value = -1e18

        for s1 in range(max_goals + 1):
            for s2 in range(max_goals + 1):

                cand_diff_1 = s1 - s2
                cand_diff_2 = s2 - s1
                cand_total = s1 + s2

                value = poisson_logpmf_v4(s1, lam1) + poisson_logpmf_v4(s2, lam2)

                o1 = candidate_outcome_label(s1, s2)
                o2 = candidate_outcome_label(s2, s1)

                # outcome support
                value += alpha * (math.log(p1[o1]) + math.log(p2[o2]))

                # goal difference support
                value -= beta_diff * (
                    abs(cand_diff_1 - d1) + abs(cand_diff_2 - d2)
                )

                # total goals support
                value -= beta_total * (
                    abs(cand_total - t1) + abs(cand_total - t2)
                )

                # football prior
                if cand_total >= 5:
                    value -= total_penalty * (cand_total - 4)

                if abs(cand_diff_1) >= 4:
                    value -= diff_penalty * (abs(cand_diff_1) - 3)

                if s1 == s2 and abs(lam1 - lam2) < 0.25:
                    value += draw_boost

                if s1 == 0 and s2 == 0 and (lam1 + lam2) > 2.25:
                    value -= 0.15

                value += common_scores.get((s1, s2), 0.0)

                if value > best_value:
                    best_value = value
                    best_score = (s1, s2)

        s1, s2 = best_score

        rows.append((r1["Id"], s1, s2))
        rows.append((r2["Id"], s2, s1))

    return pd.DataFrame(rows, columns=["Id", "team_goals", "opp_goals"])


# ============================================================
# 4. LOCAL VALIDATION
# ============================================================

cutoff_date = pd.Timestamp("2008-01-01")

hist_train = train[train["date"] < cutoff_date].copy()
val_full = train[train["date"] >= cutoff_date].copy()
val_input = val_full[BASE_COLS].copy()

print("\nHist train:", hist_train.shape)
print("Validation:", val_full.shape)
print("Validation date range:", val_full["date"].min(), "to", val_full["date"].max())

hist_fe = make_training_features(hist_train)
val_fe = make_prediction_features_from_history(
    val_input,
    hist_train[BASE_COLS + TARGET_COLS].copy()
)

feature_cols = get_feature_columns(hist_fe)
feature_cols = [c for c in feature_cols if c in val_fe.columns]

active_cat_cols = [c for c in CAT_COLS if c in feature_cols]

X_hist, medians = prepare_X(hist_fe, feature_cols, active_cat_cols)
X_val, _ = prepare_X(val_fe, feature_cols, active_cat_cols, medians=medians)

cat_idx = [X_hist.columns.get_loc(c) for c in active_cat_cols]

y_goal = hist_fe["team_goals"].clip(0, 10)
y_outcome = [
    make_outcome_label(tg, og)
    for tg, og in zip(hist_fe["team_goals"], hist_fe["opp_goals"])
]
y_diff = (hist_fe["team_goals"] - hist_fe["opp_goals"]).clip(-8, 8)
y_total = (hist_fe["team_goals"] + hist_fe["opp_goals"]).clip(0, 12)

val_outcome_true = [
    make_outcome_label(tg, og)
    for tg, og in zip(val_full["team_goals"], val_full["opp_goals"])
]

print("Number of features:", len(feature_cols))
print("Categorical features:", active_cat_cols)


# ============================================================
# 5. TRAIN VALIDATION MODELS
# ============================================================

goal_model_val = CatBoostRegressor(
    iterations=900,
    learning_rate=0.035,
    depth=6,
    loss_function="RMSE",
    eval_metric="MAE",
    l2_leaf_reg=8,
    random_seed=42,
    od_type="Iter",
    od_wait=80,
    verbose=200,
)

goal_model_val.fit(
    X_hist,
    y_goal,
    cat_features=cat_idx,
    eval_set=(X_val, val_full["team_goals"].clip(0, 10)),
    use_best_model=True,
)

outcome_model_val = CatBoostClassifier(
    iterations=700,
    learning_rate=0.04,
    depth=6,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    l2_leaf_reg=8,
    random_seed=42,
    od_type="Iter",
    od_wait=80,
    verbose=200,
)

outcome_model_val.fit(
    X_hist,
    y_outcome,
    cat_features=cat_idx,
    eval_set=(X_val, val_outcome_true),
    use_best_model=True,
)

diff_model_val = CatBoostRegressor(
    iterations=700,
    learning_rate=0.04,
    depth=6,
    loss_function="RMSE",
    eval_metric="MAE",
    l2_leaf_reg=8,
    random_seed=42,
    od_type="Iter",
    od_wait=80,
    verbose=200,
)

diff_model_val.fit(
    X_hist,
    y_diff,
    cat_features=cat_idx,
    eval_set=(X_val, (val_full["team_goals"] - val_full["opp_goals"]).clip(-8, 8)),
    use_best_model=True,
)

total_model_val = CatBoostRegressor(
    iterations=700,
    learning_rate=0.04,
    depth=6,
    loss_function="RMSE",
    eval_metric="MAE",
    l2_leaf_reg=8,
    random_seed=42,
    od_type="Iter",
    od_wait=80,
    verbose=200,
)

total_model_val.fit(
    X_hist,
    y_total,
    cat_features=cat_idx,
    eval_set=(X_val, (val_full["team_goals"] + val_full["opp_goals"]).clip(0, 12)),
    use_best_model=True,
)


# ============================================================
# 6. VALIDATION PREDICTIONS
# ============================================================

val_lambda = goal_model_val.predict(X_val)
val_outcome_probs = outcome_model_val.predict_proba(X_val)
val_diff_pred = diff_model_val.predict(X_val)
val_total_pred = total_model_val.predict(X_val)


# ============================================================
# 7. TUNE V4 PARAMS LOCALLY
# ============================================================

param_grid = []

for alpha in [0.15, 0.25, 0.35, 0.50]:
    for beta_diff in [0.08, 0.12, 0.18, 0.25]:
        for beta_total in [0.04, 0.08, 0.12]:
            param_grid.append({
                "alpha": alpha,
                "beta_diff": beta_diff,
                "beta_total": beta_total,
                "max_goals": 5,
                "total_penalty": 0.22,
                "diff_penalty": 0.50,
                "draw_boost": 0.08,
                "oneone_boost": 0.03,
            })

scores = []

for params in param_grid:
    pred = make_score_predictions_v4(
        val_fe,
        val_lambda,
        val_outcome_probs,
        val_diff_pred,
        val_total_pred,
        **params
    )

    raw_score = evaluate_prediction(val_full, pred)
    medium_score = evaluate_prediction(val_full, apply_postprocess(pred, "medium"))
    aggressive_score = evaluate_prediction(val_full, apply_postprocess(pred, "aggressive"))

    scores.append({**params, "mode": "raw", "score": raw_score})
    scores.append({**params, "mode": "medium", "score": medium_score})
    scores.append({**params, "mode": "aggressive", "score": aggressive_score})

scores_df = pd.DataFrame(scores).sort_values("score").reset_index(drop=True)

print("\nV4 TOP LOCAL SCORES:")
display(scores_df.head(20))

best = scores_df.iloc[0].to_dict()

print("\nBest V4 params:")
print(best)

best_local_score = best["score"]
best_mode = best["mode"]

print("\nBest V4 local score:", best_local_score)
print("Best V3 local score reference: 2.370148188224391")
print("Gain vs V3:", 2.370148188224391 - best_local_score)


# ============================================================
# 8. TRAIN FINAL MODELS ON ALL TRAIN
# ============================================================

all_fe = make_training_features(train)
test_fe = make_prediction_features_from_history(
    test.copy(),
    train[BASE_COLS + TARGET_COLS].copy()
)

final_feature_cols = get_feature_columns(all_fe)
final_feature_cols = [c for c in final_feature_cols if c in test_fe.columns]

final_cat_cols = [c for c in CAT_COLS if c in final_feature_cols]

X_all, final_medians = prepare_X(all_fe, final_feature_cols, final_cat_cols)
X_test, _ = prepare_X(test_fe, final_feature_cols, final_cat_cols, medians=final_medians)

final_cat_idx = [X_all.columns.get_loc(c) for c in final_cat_cols]

y_all_goal = all_fe["team_goals"].clip(0, 10)
y_all_outcome = [
    make_outcome_label(tg, og)
    for tg, og in zip(all_fe["team_goals"], all_fe["opp_goals"])
]
y_all_diff = (all_fe["team_goals"] - all_fe["opp_goals"]).clip(-8, 8)
y_all_total = (all_fe["team_goals"] + all_fe["opp_goals"]).clip(0, 12)

goal_iter = goal_model_val.get_best_iteration()
if goal_iter is None or goal_iter < 100:
    goal_iter = 500
goal_iter = int(min(max(goal_iter + 100, 500), 1200))

outcome_iter = outcome_model_val.get_best_iteration()
if outcome_iter is None or outcome_iter < 100:
    outcome_iter = 500
outcome_iter = int(min(max(outcome_iter + 100, 400), 1000))

diff_iter = diff_model_val.get_best_iteration()
if diff_iter is None or diff_iter < 100:
    diff_iter = 500
diff_iter = int(min(max(diff_iter + 100, 400), 1000))

total_iter = total_model_val.get_best_iteration()
if total_iter is None or total_iter < 100:
    total_iter = 500
total_iter = int(min(max(total_iter + 100, 400), 1000))

print("\nFinal iterations:")
print("goal:", goal_iter)
print("outcome:", outcome_iter)
print("diff:", diff_iter)
print("total:", total_iter)

goal_model_final = CatBoostRegressor(
    iterations=goal_iter,
    learning_rate=0.035,
    depth=6,
    loss_function="RMSE",
    eval_metric="MAE",
    l2_leaf_reg=8,
    random_seed=42,
    verbose=200,
)

goal_model_final.fit(
    X_all,
    y_all_goal,
    cat_features=final_cat_idx,
)

outcome_model_final = CatBoostClassifier(
    iterations=outcome_iter,
    learning_rate=0.04,
    depth=6,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    l2_leaf_reg=8,
    random_seed=42,
    verbose=200,
)

outcome_model_final.fit(
    X_all,
    y_all_outcome,
    cat_features=final_cat_idx,
)

diff_model_final = CatBoostRegressor(
    iterations=diff_iter,
    learning_rate=0.04,
    depth=6,
    loss_function="RMSE",
    eval_metric="MAE",
    l2_leaf_reg=8,
    random_seed=42,
    verbose=200,
)

diff_model_final.fit(
    X_all,
    y_all_diff,
    cat_features=final_cat_idx,
)

total_model_final = CatBoostRegressor(
    iterations=total_iter,
    learning_rate=0.04,
    depth=6,
    loss_function="RMSE",
    eval_metric="MAE",
    l2_leaf_reg=8,
    random_seed=42,
    verbose=200,
)

total_model_final.fit(
    X_all,
    y_all_total,
    cat_features=final_cat_idx,
)


# ============================================================
# 9. PREDICT TEST AND SAVE submission.csv
# ============================================================

test_lambda = goal_model_final.predict(X_test)
test_outcome_probs = outcome_model_final.predict_proba(X_test)
test_diff_pred = diff_model_final.predict(X_test)
test_total_pred = total_model_final.predict(X_test)

best_params = {
    "alpha": float(best["alpha"]),
    "beta_diff": float(best["beta_diff"]),
    "beta_total": float(best["beta_total"]),
    "max_goals": int(best["max_goals"]),
    "total_penalty": float(best["total_penalty"]),
    "diff_penalty": float(best["diff_penalty"]),
    "draw_boost": float(best["draw_boost"]),
    "oneone_boost": float(best["oneone_boost"]),
}

v4_pred = make_score_predictions_v4(
    test_fe,
    test_lambda,
    test_outcome_probs,
    test_diff_pred,
    test_total_pred,
    **best_params
)

v4_pred = sample[["Id"]].merge(v4_pred, on="Id", how="left")

v4_pred[["team_goals", "opp_goals"]] = (
    v4_pred[["team_goals", "opp_goals"]]
    .fillna(0)
    .astype(int)
)

if best_mode != "raw":
    v4_pred = apply_postprocess(v4_pred, best_mode)

# final output name only
v4_pred.to_csv("submission.csv", index=False)

print("\nSaved final file: submission.csv")
print("Best V4 local score:", best_local_score)
print("Best mode:", best_mode)
print("Best params:", best_params)

print("\nSubmission check:")
print(v4_pred.shape)
print(v4_pred.isna().sum())
print(v4_pred.head(20))

assert list(v4_pred.columns) == ["Id", "team_goals", "opp_goals"]
assert v4_pred.shape[0] == sample.shape[0]
assert v4_pred["Id"].equals(sample["Id"])
assert v4_pred[["team_goals", "opp_goals"]].isna().sum().sum() == 0

print("\nScore distribution:")
display(v4_pred[["team_goals", "opp_goals"]].value_counts().head(30))

print("\nREADY FILE: /kaggle/working/submission.csv")

Train: (78772, 47)
Test: (42422, 20)
Sample: (42422, 3)

Hist train: (69842, 47)
Validation: (8930, 47)
Validation date range: 2008-01-02 00:00:00 to 2011-08-04 00:00:00
Number of features: 79
Categorical features: ['gender', 'team', 'opponent', 'tournament', 'venue_country', 'confederation_team', 'confederation_opp']
0:	learn: 1.2599295	test: 1.2260770	best: 1.2260770 (0)	total: 73.5ms	remaining: 1m 6s
200:	learn: 1.0284548	test: 1.0177677	best: 1.0176551 (199)	total: 12.2s	remaining: 42.5s
400:	learn: 1.0129722	test: 1.0150000	best: 1.0147758 (378)	total: 24.2s	remaining: 30.1s
Stopped by overfitting detector  (80 iterations wait)

bestTest = 1.014653764
bestIteration = 413

Shrink model to first 414 iterations.
0:	learn: 1.0853138	test: 1.0851091	best: 1.0851091 (0)	total: 245ms	remaining: 2m 51s
200:	learn: 0.8704919	test: 0.8893082	best: 0.8892316 (188)	total: 46.6s	remaining: 1m 55s
400:	learn: 0.8539774	test: 0.8878987	best: 0.8877413 (387)	total: 1m 31s	remaining: 1m 8s
600:	le

,alpha,beta_diff,beta_total,max_goals,total_penalty,diff_penalty,draw_boost,oneone_boost,mode,score
0,0.50,0.25,0.12,5,0.22,0.5,0.08,0.03,raw,2.342833
1,0.35,0.18,0.12,5,0.22,0.5,0.08,0.03,raw,2.346499
2,0.50,0.12,0.12,5,0.22,0.5,0.08,0.03,raw,2.346853
3,0.35,0.25,0.12,5,0.22,0.5,0.08,0.03,raw,2.347829
4,0.50,0.18,0.12,5,0.22,0.5,0.08,0.03,raw,2.348011
5,0.25,0.18,0.04,5,0.22,0.5,0.08,0.03,raw,2.348300
6,0.25,0.12,0.12,5,0.22,0.5,0.08,0.03,raw,2.348659
7,0.25,0.25,0.04,5,0.22,0.5,0.08,0.03,raw,2.348958
8,0.35,0.12,0.12,5,0.22,0.5,0.08,0.03,raw,2.349118
9,0.35,0.18,0.08,5,0.22,0.5,0.08,0.03,raw,2.349323



Best V4 params:
{'alpha': 0.5, 'beta_diff': 0.25, 'beta_total': 0.12, 'max_goals': 5, 'total_penalty': 0.22, 'diff_penalty': 0.5, 'draw_boost': 0.08, 'oneone_boost': 0.03, 'mode': 'raw', 'score': 2.3428331357967034}

Best V4 local score: 2.3428331357967034
Best V3 local score reference: 2.370148188224391
Gain vs V3: 0.027315052427687725

Final iterations:
goal: 513
outcome: 644
diff: 400
total: 619
0:	learn: 1.2536288	total: 78.7ms	remaining: 40.3s
200:	learn: 1.0217594	total: 13.1s	remaining: 20.3s
400:	learn: 1.0064273	total: 25.7s	remaining: 7.19s
512:	learn: 1.0011503	total: 32.4s	remaining: 0us
0:	learn: 1.0847397	total: 281ms	remaining: 3m
200:	learn: 0.8683282	total: 49.1s	remaining: 1m 48s
400:	learn: 0.8523213	total: 1m 37s	remaining: 59.3s
600:	learn: 0.8414933	total: 2m 27s	remaining: 10.5s
643:	learn: 0.8395528	total: 2m 37s	remaining: 0us
0:	learn: 1.8123776	total: 88.9ms	remaining: 35.5s
200:	learn: 1.4642845	total: 13.2s	remaining: 13s
399:	learn: 1.4364240	total: 25.9s

team_goals  opp_goals
1           1            5942
0           1            4980
1           0            4980
            2            4751
2           1            4751
0           2            4385
2           0            4385
3           0            2420
0           3            2420
            4             732
4           0             732
1           3             602
3           1             602
0           5             290
5           0             290
4           1              70
1           4              70
2           3               7
3           2               7
1           5               2
2           2               2
5           1               2
Name: count, dtype: int64


READY FILE: /kaggle/working/submission.csv


VERSI 4 B YAA 

In [5]:
# ============================================================
# V4B: CALIBRATION SWEEP ONLY, NO RETRAIN
# Final output: submission.csv
# Run AFTER V4 is finished
# ============================================================

import pandas as pd
import numpy as np

# ============================================================
# 1. SAFETY CHECK
# ============================================================

required_vars = [
    "val_fe",
    "val_lambda",
    "val_outcome_probs",
    "val_diff_pred",
    "val_total_pred",
    "val_full",
    "test_fe",
    "test_lambda",
    "test_outcome_probs",
    "test_diff_pred",
    "test_total_pred",
    "sample",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise Exception("Missing variables from V4. Run V4 first. Missing: " + str(missing))

required_funcs = [
    "make_score_predictions_v4",
    "evaluate_prediction",
    "apply_postprocess",
]

missing_f = [f for f in required_funcs if f not in globals()]
if missing_f:
    raise Exception("Missing functions from V4. Run V4 first. Missing: " + str(missing_f))


# ============================================================
# 2. EXTRA POST-PROCESSING FOR HIGH SCORES
# ============================================================

def cap_extreme_scores(sub, mode="none"):
    """
    mode:
    - none: no change
    - cap5: reduce only 5-goal scores
    - cap4: reduce 4/5-goal scores moderately
    - cap_strict: reduce 4/5-goal scores aggressively
    """
    out = sub.copy()

    def transform(tg, og):
        tg, og = int(tg), int(og)

        if mode == "none":
            return tg, og

        # Keep draw simple
        if tg == og:
            if tg >= 3:
                return 2, 2
            return tg, og

        # Team wins
        if tg > og:
            if mode == "cap5":
                if tg >= 5:
                    return 4, max(0, min(1, og))
                return tg, og

            if mode == "cap4":
                if tg >= 5:
                    return 4, max(0, min(1, og))
                if tg == 4 and og == 0:
                    return 3, 0
                return tg, og

            if mode == "cap_strict":
                if tg >= 5:
                    return 3, 0 if og == 0 else 3, 1
                if tg == 4:
                    return 3, 0 if og == 0 else 3, 1
                return tg, og

        # Opponent wins
        if og > tg:
            if mode == "cap5":
                if og >= 5:
                    return max(0, min(1, tg)), 4
                return tg, og

            if mode == "cap4":
                if og >= 5:
                    return max(0, min(1, tg)), 4
                if og == 4 and tg == 0:
                    return 0, 3
                return tg, og

            if mode == "cap_strict":
                if og >= 5:
                    return 0 if tg == 0 else 1, 3
                if og == 4:
                    return 0 if tg == 0 else 1, 3
                return tg, og

        return tg, og

    pairs = out.apply(lambda r: transform(r["team_goals"], r["opp_goals"]), axis=1)
    out["team_goals"] = [p[0] for p in pairs]
    out["opp_goals"] = [p[1] for p in pairs]

    return out


# ============================================================
# 3. PARAMETER SWEEP
# ============================================================

param_grid = []

for alpha in [0.35, 0.50, 0.65]:
    for beta_diff in [0.18, 0.25, 0.32]:
        for beta_total in [0.08, 0.12, 0.16]:
            for total_penalty in [0.22, 0.35, 0.50]:
                for diff_penalty in [0.50, 0.70]:
                    for max_goals in [4, 5]:
                        param_grid.append({
                            "alpha": alpha,
                            "beta_diff": beta_diff,
                            "beta_total": beta_total,
                            "max_goals": max_goals,
                            "total_penalty": total_penalty,
                            "diff_penalty": diff_penalty,
                            "draw_boost": 0.08,
                            "oneone_boost": 0.03,
                        })

results = []

for params in param_grid:
    pred = make_score_predictions_v4(
        val_fe,
        val_lambda,
        val_outcome_probs,
        val_diff_pred,
        val_total_pred,
        **params
    )

    for post_mode in ["raw", "medium", "aggressive"]:
        if post_mode == "raw":
            base_pred = pred.copy()
        else:
            base_pred = apply_postprocess(pred, post_mode)

        for cap_mode in ["none", "cap5", "cap4", "cap_strict"]:
            final_pred = cap_extreme_scores(base_pred, cap_mode)
            score = evaluate_prediction(val_full, final_pred)

            results.append({
                **params,
                "post_mode": post_mode,
                "cap_mode": cap_mode,
                "score": score,
            })

results_df = pd.DataFrame(results).sort_values("score").reset_index(drop=True)

print("V4B TOP LOCAL SCORES:")
display(results_df.head(30))

best = results_df.iloc[0].to_dict()

print("\nBest V4B config:")
print(best)

print("\nReference scores:")
print("V3 local:", 2.370148188224391)
print("V4 raw local:", 2.3428331357967034)
print("V4B best local:", best["score"])
print("Gain vs V3:", 2.370148188224391 - best["score"])
print("Gain vs V4 raw:", 2.3428331357967034 - best["score"])


# ============================================================
# 4. CREATE FINAL TEST SUBMISSION.CSV
# ============================================================

best_params = {
    "alpha": float(best["alpha"]),
    "beta_diff": float(best["beta_diff"]),
    "beta_total": float(best["beta_total"]),
    "max_goals": int(best["max_goals"]),
    "total_penalty": float(best["total_penalty"]),
    "diff_penalty": float(best["diff_penalty"]),
    "draw_boost": float(best["draw_boost"]),
    "oneone_boost": float(best["oneone_boost"]),
}

test_pred = make_score_predictions_v4(
    test_fe,
    test_lambda,
    test_outcome_probs,
    test_diff_pred,
    test_total_pred,
    **best_params
)

test_pred = sample[["Id"]].merge(test_pred, on="Id", how="left")

test_pred[["team_goals", "opp_goals"]] = (
    test_pred[["team_goals", "opp_goals"]]
    .fillna(0)
    .astype(int)
)

if best["post_mode"] != "raw":
    test_pred = apply_postprocess(test_pred, best["post_mode"])

test_pred = cap_extreme_scores(test_pred, best["cap_mode"])

test_pred.to_csv("submission.csv", index=False)

print("\nSaved final file: submission.csv")
print("Best V4B local score:", best["score"])
print("Best params:", best_params)
print("Post mode:", best["post_mode"])
print("Cap mode:", best["cap_mode"])

print("\nSubmission check:")
print(test_pred.shape)
print(test_pred.isna().sum())
print(test_pred.head(20))

assert list(test_pred.columns) == ["Id", "team_goals", "opp_goals"]
assert test_pred.shape[0] == sample.shape[0]
assert test_pred["Id"].equals(sample["Id"])
assert test_pred[["team_goals", "opp_goals"]].isna().sum().sum() == 0

print("\nScore distribution:")
display(test_pred[["team_goals", "opp_goals"]].value_counts().head(30))

print("\nREADY FILE: /kaggle/working/submission.csv")

V4B TOP LOCAL SCORES:


,alpha,beta_diff,beta_total,max_goals,total_penalty,diff_penalty,draw_boost,oneone_boost,post_mode,cap_mode,score
0,0.50,0.32,0.16,5,0.50,0.5,0.08,0.03,raw,none,2.338160
1,0.50,0.32,0.16,5,0.35,0.5,0.08,0.03,raw,none,2.339318
2,0.65,0.32,0.16,5,0.50,0.5,0.08,0.03,raw,none,2.339450
3,0.50,0.32,0.16,4,0.50,0.5,0.08,0.03,raw,cap5,2.339715
4,0.50,0.32,0.16,5,0.50,0.5,0.08,0.03,raw,cap5,2.339715
5,0.50,0.32,0.16,4,0.50,0.5,0.08,0.03,raw,none,2.339715
6,0.50,0.32,0.16,5,0.22,0.5,0.08,0.03,raw,none,2.340133
7,0.65,0.32,0.16,5,0.35,0.5,0.08,0.03,raw,none,2.340607
8,0.50,0.32,0.16,4,0.35,0.5,0.08,0.03,raw,cap5,2.340626
9,0.50,0.32,0.16,4,0.35,0.5,0.08,0.03,raw,none,2.340626



Best V4B config:
{'alpha': 0.5, 'beta_diff': 0.32, 'beta_total': 0.16, 'max_goals': 5, 'total_penalty': 0.5, 'diff_penalty': 0.5, 'draw_boost': 0.08, 'oneone_boost': 0.03, 'post_mode': 'raw', 'cap_mode': 'none', 'score': 2.3381604966141163}

Reference scores:
V3 local: 2.370148188224391
V4 raw local: 2.3428331357967034
V4B best local: 2.3381604966141163
Gain vs V3: 0.031987691610274815
Gain vs V4 raw: 0.0046726391825870905

Saved final file: submission.csv
Best V4B local score: 2.3381604966141163
Best params: {'alpha': 0.5, 'beta_diff': 0.32, 'beta_total': 0.16, 'max_goals': 5, 'total_penalty': 0.5, 'diff_penalty': 0.5, 'draw_boost': 0.08, 'oneone_boost': 0.03}
Post mode: raw
Cap mode: none

Submission check:
(42422, 3)
Id            0
team_goals    0
opp_goals     0
dtype: int64
                        Id  team_goals  opp_goals
0       M034984_Seychelles           2          1
1        M034984_Mauritius           1          2
2          M034985_Comoros           1          1
3       

team_goals  opp_goals
1           1            6464
            2            5380
2           1            5380
0           2            4151
2           0            4151
1           0            4144
0           1            4144
            3            2392
3           0            2392
0           4             852
4           0             852
1           3             757
3           1             757
0           5             257
5           0             257
1           4              39
4           1              39
2           2               6
            3               4
3           2               4
Name: count, dtype: int64


READY FILE: /kaggle/working/submission.csv


In [6]:
# ============================================================
# V6: FINE RANDOM SEARCH AROUND V4B
# NO RETRAIN
# Only overwrite submission.csv if local improves enough
# ============================================================

import os
import shutil
import numpy as np
import pandas as pd
from IPython.display import display

# Backup current file, should be V4B
if os.path.exists("/kaggle/working/submission.csv"):
    shutil.copy("/kaggle/working/submission.csv", "/kaggle/working/submission_v4b_backup_v6.csv")
    print("Backup saved: submission_v4b_backup_v6.csv")

V4B_LOCAL_SCORE = 2.3381604966141163
PUBLIC_BEST = 2.958113509

required_vars = [
    "val_fe", "val_lambda", "val_outcome_probs", "val_diff_pred", "val_total_pred", "val_full",
    "test_fe", "test_lambda", "test_outcome_probs", "test_diff_pred", "test_total_pred", "sample"
]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise Exception("Missing variables from V4B. Run V4/V4B first. Missing: " + str(missing))

required_funcs = ["make_score_predictions_v4", "evaluate_prediction", "apply_postprocess"]
missing_f = [f for f in required_funcs if f not in globals()]
if missing_f:
    raise Exception("Missing functions from V4B. Missing: " + str(missing_f))


# ============================================================
# 1. EXTRA HIGH-SCORE CONTROL
# ============================================================

def cap_by_mode(sub, mode="none"):
    out = sub.copy()

    def fix(tg, og):
        tg, og = int(tg), int(og)

        if mode == "none":
            return tg, og

        if tg == og:
            if mode in ["cap_draw2", "strict"] and tg >= 3:
                return 2, 2
            return tg, og

        if tg > og:
            if mode == "cap5":
                if tg >= 5:
                    return 4, min(1, og)
            elif mode == "cap4":
                if tg >= 5:
                    return 4, min(1, og)
                if tg == 4 and og == 0:
                    return 3, 0
            elif mode == "strict":
                if tg >= 4:
                    return 3, 0 if og == 0 else 3, 1
            return tg, og

        if og > tg:
            if mode == "cap5":
                if og >= 5:
                    return min(1, tg), 4
            elif mode == "cap4":
                if og >= 5:
                    return min(1, tg), 4
                if og == 4 and tg == 0:
                    return 0, 3
            elif mode == "strict":
                if og >= 4:
                    return 0 if tg == 0 else 1, 3
            return tg, og

        return tg, og

    pairs = out.apply(lambda r: fix(r["team_goals"], r["opp_goals"]), axis=1)
    out["team_goals"] = [p[0] for p in pairs]
    out["opp_goals"] = [p[1] for p in pairs]
    return out


# ============================================================
# 2. RANDOM + LOCAL GRID SEARCH
# ============================================================

np.random.seed(2026)

configs = []

# deterministic fine grid around V4B best
for alpha in [0.44, 0.48, 0.50, 0.52, 0.56, 0.60]:
    for beta_diff in [0.26, 0.30, 0.32, 0.34, 0.38]:
        for beta_total in [0.12, 0.14, 0.16, 0.18, 0.20]:
            for total_penalty in [0.42, 0.50, 0.58, 0.66]:
                configs.append({
                    "alpha": alpha,
                    "beta_diff": beta_diff,
                    "beta_total": beta_total,
                    "max_goals": 5,
                    "total_penalty": total_penalty,
                    "diff_penalty": 0.50,
                    "draw_boost": 0.08,
                    "oneone_boost": 0.03,
                })

# random search for more exploration
for _ in range(250):
    configs.append({
        "alpha": float(np.random.uniform(0.40, 0.68)),
        "beta_diff": float(np.random.uniform(0.22, 0.42)),
        "beta_total": float(np.random.uniform(0.10, 0.24)),
        "max_goals": 5,
        "total_penalty": float(np.random.uniform(0.35, 0.75)),
        "diff_penalty": float(np.random.uniform(0.40, 0.80)),
        "draw_boost": float(np.random.choice([0.04, 0.06, 0.08, 0.10, 0.12])),
        "oneone_boost": float(np.random.choice([0.00, 0.02, 0.03, 0.05, 0.07])),
    })

results = []

for i, cfg in enumerate(configs):
    pred = make_score_predictions_v4(
        val_fe,
        val_lambda,
        val_outcome_probs,
        val_diff_pred,
        val_total_pred,
        **cfg
    )

    for post_mode in ["raw", "medium", "aggressive"]:
        if post_mode == "raw":
            p0 = pred
        else:
            p0 = apply_postprocess(pred, post_mode)

        for cap_mode in ["none", "cap5", "cap4", "strict"]:
            p1 = cap_by_mode(p0, cap_mode)
            score = evaluate_prediction(val_full, p1)

            results.append({
                **cfg,
                "post_mode": post_mode,
                "cap_mode": cap_mode,
                "score": score
            })

    if (i + 1) % 100 == 0:
        print(f"Checked {i+1}/{len(configs)} configs")

v6_df = pd.DataFrame(results).sort_values("score").reset_index(drop=True)

print("\nV6 TOP LOCAL SCORES:")
display(v6_df.head(30))

best_v6 = v6_df.iloc[0].to_dict()

print("\nBest V6 config:")
print(best_v6)

print("\nReference:")
print("V4B local:", V4B_LOCAL_SCORE)
print("V6 best local:", best_v6["score"])
print("Gain vs V4B:", V4B_LOCAL_SCORE - best_v6["score"])


# ============================================================
# 3. CREATE FINAL ONLY IF BETTER
# ============================================================

MIN_REQUIRED_GAIN = 0.008

if best_v6["score"] < V4B_LOCAL_SCORE - MIN_REQUIRED_GAIN:
    print("\nV6 improved enough. Creating new submission.csv")

    best_params = {
        "alpha": float(best_v6["alpha"]),
        "beta_diff": float(best_v6["beta_diff"]),
        "beta_total": float(best_v6["beta_total"]),
        "max_goals": int(best_v6["max_goals"]),
        "total_penalty": float(best_v6["total_penalty"]),
        "diff_penalty": float(best_v6["diff_penalty"]),
        "draw_boost": float(best_v6["draw_boost"]),
        "oneone_boost": float(best_v6["oneone_boost"]),
    }

    test_pred = make_score_predictions_v4(
        test_fe,
        test_lambda,
        test_outcome_probs,
        test_diff_pred,
        test_total_pred,
        **best_params
    )

    test_pred = sample[["Id"]].merge(test_pred, on="Id", how="left")
    test_pred[["team_goals", "opp_goals"]] = (
        test_pred[["team_goals", "opp_goals"]]
        .fillna(0)
        .astype(int)
    )

    if best_v6["post_mode"] != "raw":
        test_pred = apply_postprocess(test_pred, best_v6["post_mode"])

    test_pred = cap_by_mode(test_pred, best_v6["cap_mode"])
    test_pred.to_csv("submission.csv", index=False)

    final_sub = test_pred.copy()
    final_version = "V6"

else:
    print("\nV6 did NOT improve enough. Restoring V4B submission.csv")

    if os.path.exists("/kaggle/working/submission_v4b_backup_v6.csv"):
        shutil.copy("/kaggle/working/submission_v4b_backup_v6.csv", "/kaggle/working/submission.csv")

    final_sub = pd.read_csv("/kaggle/working/submission.csv")
    final_version = "V4B_RESTORED"


# ============================================================
# 4. FINAL CHECK
# ============================================================

print("\nFinal version in submission.csv:", final_version)
print("Saved final file: submission.csv")

print("\nSubmission check:")
print(final_sub.shape)
print(final_sub.isna().sum())
print(final_sub.head(20))

assert list(final_sub.columns) == ["Id", "team_goals", "opp_goals"]
assert final_sub.shape[0] == sample.shape[0]
assert final_sub["Id"].equals(sample["Id"])
assert final_sub[["team_goals", "opp_goals"]].isna().sum().sum() == 0

print("\nScore distribution:")
display(final_sub[["team_goals", "opp_goals"]].value_counts().head(30))

print("\nREADY FILE: /kaggle/working/submission.csv")

Backup saved: submission_v4b_backup_v6.csv
Checked 100/850 configs
Checked 200/850 configs
Checked 300/850 configs
Checked 400/850 configs
Checked 500/850 configs
Checked 600/850 configs
Checked 700/850 configs
Checked 800/850 configs

V6 TOP LOCAL SCORES:


,alpha,beta_diff,beta_total,max_goals,total_penalty,diff_penalty,draw_boost,oneone_boost,post_mode,cap_mode,score
0,0.556488,0.326552,0.191935,5,0.592817,0.536985,0.04,0.03,raw,none,2.335596
1,0.600881,0.254116,0.148455,5,0.378762,0.505649,0.04,0.03,raw,none,2.336020
2,0.560000,0.340000,0.180000,5,0.580000,0.500000,0.08,0.03,raw,none,2.336127
3,0.600000,0.340000,0.180000,5,0.580000,0.500000,0.08,0.03,raw,none,2.336153
4,0.520000,0.340000,0.180000,5,0.580000,0.500000,0.08,0.03,raw,none,2.336235
5,0.510990,0.383922,0.183752,5,0.749933,0.464961,0.10,0.00,raw,none,2.336279
6,0.526647,0.369744,0.172330,5,0.664931,0.758104,0.12,0.02,raw,cap5,2.336335
7,0.526647,0.369744,0.172330,5,0.664931,0.758104,0.12,0.02,raw,none,2.336335
8,0.480000,0.320000,0.180000,5,0.580000,0.500000,0.08,0.03,raw,none,2.336522
9,0.560000,0.320000,0.180000,5,0.580000,0.500000,0.08,0.03,raw,none,2.336878



Best V6 config:
{'alpha': 0.5564880990529631, 'beta_diff': 0.3265516485119264, 'beta_total': 0.19193477605870235, 'max_goals': 5, 'total_penalty': 0.5928166608611491, 'diff_penalty': 0.5369851071102402, 'draw_boost': 0.04, 'oneone_boost': 0.03, 'post_mode': 'raw', 'cap_mode': 'none', 'score': 2.335596315144487}

Reference:
V4B local: 2.3381604966141163
V6 best local: 2.335596315144487
Gain vs V4B: 0.002564181469629112

V6 did NOT improve enough. Restoring V4B submission.csv

Final version in submission.csv: V4B_RESTORED
Saved final file: submission.csv

Submission check:
(42422, 3)
Id            0
team_goals    0
opp_goals     0
dtype: int64
                        Id  team_goals  opp_goals
0       M034984_Seychelles           2          1
1        M034984_Mauritius           1          2
2          M034985_Comoros           1          1
3         M034985_Maldives           1          1
4          M034986_Réunion           2          1
5       M034986_Madagascar           1          2

team_goals  opp_goals
1           1            6464
            2            5380
2           1            5380
0           2            4151
2           0            4151
1           0            4144
0           1            4144
            3            2392
3           0            2392
0           4             852
4           0             852
1           3             757
3           1             757
0           5             257
5           0             257
1           4              39
4           1              39
2           2               6
            3               4
3           2               4
Name: count, dtype: int64


READY FILE: /kaggle/working/submission.csv


In [7]:
# ============================================================
# V7: TOURNAMENT-SPECIFIC CALIBRATION
# NO RETRAIN
# Only overwrite submission.csv if local improves enough
# ============================================================

import os
import shutil
import numpy as np
import pandas as pd
from IPython.display import display

# ============================================================
# 0. BACKUP CURRENT FILE, SHOULD BE V4B
# ============================================================

if os.path.exists("/kaggle/working/submission.csv"):
    shutil.copy(
        "/kaggle/working/submission.csv",
        "/kaggle/working/submission_v4b_backup_v7.csv"
    )
    print("Backup saved: submission_v4b_backup_v7.csv")

V4B_LOCAL_SCORE = 2.3381604966141163
PUBLIC_BEST = 2.958113509

required_vars = [
    "val_fe", "val_lambda", "val_outcome_probs", "val_diff_pred", "val_total_pred", "val_full",
    "test_fe", "test_lambda", "test_outcome_probs", "test_diff_pred", "test_total_pred", "sample"
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise Exception("Missing variables from V4/V4B. Run V4/V4B first. Missing: " + str(missing))

required_funcs = ["make_score_predictions_v4", "evaluate_prediction", "apply_postprocess"]

missing_f = [f for f in required_funcs if f not in globals()]
if missing_f:
    raise Exception("Missing functions from V4/V4B. Missing: " + str(missing_f))


# ============================================================
# 1. TOURNAMENT GROUPING
# ============================================================

def get_tournament_group(tournament):
    t = str(tournament).lower()

    if "friendly" in t:
        return "friendly"

    if "qualification" in t or "qualifier" in t or "qualifying" in t:
        return "qualification"

    major_keywords = [
        "fifa world cup",
        "uefa euro",
        "copa américa",
        "copa america",
        "african cup",
        "africa cup",
        "asian cup",
        "gold cup",
        "confederations cup",
        "nations league",
        "olympic",
        "championship",
        "cup",
    ]

    if any(k in t for k in major_keywords):
        return "cup_or_championship"

    return "other"


def add_v7_group(df):
    out = df.copy()
    out["v7_group"] = out["tournament"].apply(get_tournament_group)
    return out


val_fe_v7 = add_v7_group(val_fe)
test_fe_v7 = add_v7_group(test_fe)
val_full_v7 = add_v7_group(val_full)

print("Validation group counts:")
display(val_fe_v7["v7_group"].value_counts())

print("Test group counts:")
display(test_fe_v7["v7_group"].value_counts())


# ============================================================
# 2. HIGH SCORE CAP
# ============================================================

def cap_by_mode_v7(sub, mode="none"):
    out = sub.copy()

    def fix(tg, og):
        tg, og = int(tg), int(og)

        if mode == "none":
            return tg, og

        if tg == og:
            if mode in ["draw2", "strict"] and tg >= 3:
                return 2, 2
            return tg, og

        if tg > og:
            if mode == "cap5":
                if tg >= 5:
                    return 4, min(1, og)
            elif mode == "cap4":
                if tg >= 5:
                    return 4, min(1, og)
                if tg == 4 and og == 0:
                    return 3, 0
            elif mode == "strict":
                if tg >= 4:
                    return 3, 0 if og == 0 else 3, 1
            return tg, og

        if og > tg:
            if mode == "cap5":
                if og >= 5:
                    return min(1, tg), 4
            elif mode == "cap4":
                if og >= 5:
                    return min(1, tg), 4
                if og == 4 and tg == 0:
                    return 0, 3
            elif mode == "strict":
                if og >= 4:
                    return 0 if tg == 0 else 1, 3
            return tg, og

        return tg, og

    pairs = out.apply(lambda r: fix(r["team_goals"], r["opp_goals"]), axis=1)
    out["team_goals"] = [p[0] for p in pairs]
    out["opp_goals"] = [p[1] for p in pairs]
    return out


# ============================================================
# 3. CONFIG CANDIDATES
# ============================================================

base_configs = [
    # V4B public-success config
    {
        "name": "v4b",
        "alpha": 0.50,
        "beta_diff": 0.32,
        "beta_total": 0.16,
        "max_goals": 5,
        "total_penalty": 0.50,
        "diff_penalty": 0.50,
        "draw_boost": 0.08,
        "oneone_boost": 0.03,
    },
    # V6 best local config, but not enough global gain
    {
        "name": "v6_local",
        "alpha": 0.5564880990529631,
        "beta_diff": 0.3265516485119264,
        "beta_total": 0.19193477605870235,
        "max_goals": 5,
        "total_penalty": 0.5928166608611491,
        "diff_penalty": 0.5369851071102402,
        "draw_boost": 0.04,
        "oneone_boost": 0.03,
    },
    # slightly safer
    {
        "name": "safe_total",
        "alpha": 0.52,
        "beta_diff": 0.34,
        "beta_total": 0.20,
        "max_goals": 5,
        "total_penalty": 0.66,
        "diff_penalty": 0.60,
        "draw_boost": 0.06,
        "oneone_boost": 0.03,
    },
    # more outcome-driven
    {
        "name": "outcome_plus",
        "alpha": 0.62,
        "beta_diff": 0.32,
        "beta_total": 0.16,
        "max_goals": 5,
        "total_penalty": 0.50,
        "diff_penalty": 0.50,
        "draw_boost": 0.06,
        "oneone_boost": 0.02,
    },
    # more draw support
    {
        "name": "draw_support",
        "alpha": 0.45,
        "beta_diff": 0.28,
        "beta_total": 0.14,
        "max_goals": 5,
        "total_penalty": 0.50,
        "diff_penalty": 0.50,
        "draw_boost": 0.12,
        "oneone_boost": 0.07,
    },
    # aggressive diff
    {
        "name": "diff_plus",
        "alpha": 0.50,
        "beta_diff": 0.40,
        "beta_total": 0.18,
        "max_goals": 5,
        "total_penalty": 0.56,
        "diff_penalty": 0.60,
        "draw_boost": 0.06,
        "oneone_boost": 0.03,
    },
]

post_modes = ["raw"]
cap_modes = ["none", "cap5", "cap4"]


# ============================================================
# 4. PREDICT BY GROUP
# ============================================================

def predict_with_config_for_mask(feature_df, lambdas, outcome_probs, diff_preds, total_preds, mask, config):
    idx = np.where(mask)[0]
    f_sub = feature_df.iloc[idx].copy()

    pred = make_score_predictions_v4(
        f_sub,
        np.asarray(lambdas)[idx],
        np.asarray(outcome_probs)[idx],
        np.asarray(diff_preds)[idx],
        np.asarray(total_preds)[idx],
        alpha=float(config["alpha"]),
        beta_diff=float(config["beta_diff"]),
        beta_total=float(config["beta_total"]),
        max_goals=int(config["max_goals"]),
        total_penalty=float(config["total_penalty"]),
        diff_penalty=float(config["diff_penalty"]),
        draw_boost=float(config["draw_boost"]),
        oneone_boost=float(config["oneone_boost"]),
    )

    return pred


def evaluate_group_config(group_name, config, post_mode="raw", cap_mode="none"):
    mask = (val_fe_v7["v7_group"].values == group_name)

    pred = predict_with_config_for_mask(
        val_fe_v7,
        val_lambda,
        val_outcome_probs,
        val_diff_pred,
        val_total_pred,
        mask,
        config,
    )

    if post_mode != "raw":
        pred = apply_postprocess(pred, post_mode)

    pred = cap_by_mode_v7(pred, cap_mode)

    score = evaluate_prediction(val_full_v7.loc[mask].copy(), pred)

    return score


# ============================================================
# 5. SELECT BEST CONFIG PER GROUP
# ============================================================

group_best = {}
group_result_rows = []

groups = list(val_fe_v7["v7_group"].value_counts().index)

MIN_ROWS_PER_GROUP = 300

for group_name in groups:
    n_rows = int((val_fe_v7["v7_group"] == group_name).sum())

    if n_rows < MIN_ROWS_PER_GROUP:
        group_best[group_name] = {
            **base_configs[0],
            "post_mode": "raw",
            "cap_mode": "none",
            "group_score": None,
            "n_rows": n_rows,
        }
        continue

    best_score = 1e18
    best_conf = None

    for cfg in base_configs:
        for post_mode in post_modes:
            for cap_mode in cap_modes:
                score = evaluate_group_config(group_name, cfg, post_mode, cap_mode)

                row = {
                    "group": group_name,
                    "n_rows": n_rows,
                    "config_name": cfg["name"],
                    "post_mode": post_mode,
                    "cap_mode": cap_mode,
                    "score": score,
                    **{k: v for k, v in cfg.items() if k != "name"},
                }
                group_result_rows.append(row)

                if score < best_score:
                    best_score = score
                    best_conf = {
                        **cfg,
                        "post_mode": post_mode,
                        "cap_mode": cap_mode,
                        "group_score": score,
                        "n_rows": n_rows,
                    }

    group_best[group_name] = best_conf

group_results_df = pd.DataFrame(group_result_rows).sort_values(["group", "score"])

print("\nV7 GROUP CONFIG RESULTS:")
display(group_results_df.groupby("group").head(5))

print("\nBest config per group:")
for g, cfg in group_best.items():
    print(g, cfg)


# ============================================================
# 6. COMBINE VALIDATION PREDICTIONS USING GROUP-SPECIFIC CONFIG
# ============================================================

val_pred_parts = []

for group_name, cfg in group_best.items():
    mask = (val_fe_v7["v7_group"].values == group_name)

    if mask.sum() == 0:
        continue

    pred = predict_with_config_for_mask(
        val_fe_v7,
        val_lambda,
        val_outcome_probs,
        val_diff_pred,
        val_total_pred,
        mask,
        cfg,
    )

    if cfg["post_mode"] != "raw":
        pred = apply_postprocess(pred, cfg["post_mode"])

    pred = cap_by_mode_v7(pred, cfg["cap_mode"])

    val_pred_parts.append(pred)

v7_val_pred = pd.concat(val_pred_parts, axis=0, ignore_index=True)

# align to validation IDs
v7_val_pred = val_full_v7[["Id"]].merge(v7_val_pred, on="Id", how="left")
v7_val_pred[["team_goals", "opp_goals"]] = (
    v7_val_pred[["team_goals", "opp_goals"]]
    .fillna(0)
    .astype(int)
)

v7_local_score = evaluate_prediction(val_full_v7, v7_val_pred)

print("\nV7 local score:", v7_local_score)
print("V4B local score:", V4B_LOCAL_SCORE)
print("Gain vs V4B:", V4B_LOCAL_SCORE - v7_local_score)


# ============================================================
# 7. CREATE TEST SUBMISSION ONLY IF IMPROVED ENOUGH
# ============================================================

MIN_REQUIRED_GAIN = 0.010

if v7_local_score < V4B_LOCAL_SCORE - MIN_REQUIRED_GAIN:
    print("\nV7 improved enough. Creating new submission.csv")

    test_pred_parts = []

    for group_name, cfg in group_best.items():
        mask = (test_fe_v7["v7_group"].values == group_name)

        if mask.sum() == 0:
            continue

        pred = predict_with_config_for_mask(
            test_fe_v7,
            test_lambda,
            test_outcome_probs,
            test_diff_pred,
            test_total_pred,
            mask,
            cfg,
        )

        if cfg["post_mode"] != "raw":
            pred = apply_postprocess(pred, cfg["post_mode"])

        pred = cap_by_mode_v7(pred, cfg["cap_mode"])

        test_pred_parts.append(pred)

    v7_test_pred = pd.concat(test_pred_parts, axis=0, ignore_index=True)
    v7_test_pred = sample[["Id"]].merge(v7_test_pred, on="Id", how="left")

    v7_test_pred[["team_goals", "opp_goals"]] = (
        v7_test_pred[["team_goals", "opp_goals"]]
        .fillna(0)
        .astype(int)
    )

    v7_test_pred.to_csv("submission.csv", index=False)

    final_sub = v7_test_pred.copy()
    final_version = "V7"

else:
    print("\nV7 did NOT improve enough. Restoring V4B submission.csv")

    if os.path.exists("/kaggle/working/submission_v4b_backup_v7.csv"):
        shutil.copy(
            "/kaggle/working/submission_v4b_backup_v7.csv",
            "/kaggle/working/submission.csv"
        )

    final_sub = pd.read_csv("/kaggle/working/submission.csv")
    final_version = "V4B_RESTORED"


# ============================================================
# 8. FINAL CHECK
# ============================================================

print("\nFinal version in submission.csv:", final_version)
print("Saved final file: submission.csv")

print("\nSubmission check:")
print(final_sub.shape)
print(final_sub.isna().sum())
print(final_sub.head(20))

assert list(final_sub.columns) == ["Id", "team_goals", "opp_goals"]
assert final_sub.shape[0] == sample.shape[0]
assert final_sub["Id"].equals(sample["Id"])
assert final_sub[["team_goals", "opp_goals"]].isna().sum().sum() == 0

print("\nScore distribution:")
display(final_sub[["team_goals", "opp_goals"]].value_counts().head(30))

print("\nREADY FILE: /kaggle/working/submission.csv")

Backup saved: submission_v4b_backup_v7.csv
Validation group counts:


v7_group
qualification          3380
friendly               2734
cup_or_championship    2324
other                   492
Name: count, dtype: int64

Test group counts:


v7_group
qualification          16406
cup_or_championship    12636
friendly               10998
other                   2382
Name: count, dtype: int64


V7 GROUP CONFIG RESULTS:


,group,n_rows,config_name,post_mode,cap_mode,score,alpha,beta_diff,beta_total,max_goals,total_penalty,diff_penalty,draw_boost,oneone_boost
45,cup_or_championship,2324,outcome_plus,raw,none,2.472441,0.620000,0.320000,0.160000,5,0.500000,0.500000,0.06,0.02
47,cup_or_championship,2324,outcome_plus,raw,cap4,2.473548,0.620000,0.320000,0.160000,5,0.500000,0.500000,0.06,0.02
46,cup_or_championship,2324,outcome_plus,raw,cap5,2.475081,0.620000,0.320000,0.160000,5,0.500000,0.500000,0.06,0.02
36,cup_or_championship,2324,v4b,raw,none,2.477019,0.500000,0.320000,0.160000,5,0.500000,0.500000,0.08,0.03
38,cup_or_championship,2324,v4b,raw,cap4,2.478127,0.500000,0.320000,0.160000,5,0.500000,0.500000,0.08,0.03
23,friendly,2734,v6_local,raw,cap4,2.190294,0.556488,0.326552,0.191935,5,0.592817,0.536985,0.04,0.03
21,friendly,2734,v6_local,raw,none,2.190459,0.556488,0.326552,0.191935,5,0.592817,0.536985,0.04,0.03
22,friendly,2734,v6_local,raw,cap5,2.190459,0.556488,0.326552,0.191935,5,0.592817,0.536985,0.04,0.03
20,friendly,2734,v4b,raw,cap4,2.194471,0.500000,0.320000,0.160000,5,0.500000,0.500000,0.08,0.03
18,friendly,2734,v4b,raw,none,2.194635,0.500000,0.320000,0.160000,5,0.500000,0.500000,0.08,0.03



Best config per group:
qualification {'name': 'v6_local', 'alpha': 0.5564880990529631, 'beta_diff': 0.3265516485119264, 'beta_total': 0.19193477605870235, 'max_goals': 5, 'total_penalty': 0.5928166608611491, 'diff_penalty': 0.5369851071102402, 'draw_boost': 0.04, 'oneone_boost': 0.03, 'post_mode': 'raw', 'cap_mode': 'none', 'group_score': np.float64(2.2447032599461934), 'n_rows': 3380}
friendly {'name': 'v6_local', 'alpha': 0.5564880990529631, 'beta_diff': 0.3265516485119264, 'beta_total': 0.19193477605870235, 'max_goals': 5, 'total_penalty': 0.5928166608611491, 'diff_penalty': 0.5369851071102402, 'draw_boost': 0.04, 'oneone_boost': 0.03, 'post_mode': 'raw', 'cap_mode': 'cap4', 'group_score': np.float64(2.1902944102668895), 'n_rows': 2734}
cup_or_championship {'name': 'outcome_plus', 'alpha': 0.62, 'beta_diff': 0.32, 'beta_total': 0.16, 'max_goals': 5, 'total_penalty': 0.5, 'diff_penalty': 0.5, 'draw_boost': 0.06, 'oneone_boost': 0.02, 'post_mode': 'raw', 'cap_mode': 'none', 'group_sc

team_goals  opp_goals
1           1            6464
            2            5380
2           1            5380
0           2            4151
2           0            4151
1           0            4144
0           1            4144
            3            2392
3           0            2392
0           4             852
4           0             852
1           3             757
3           1             757
0           5             257
5           0             257
1           4              39
4           1              39
2           2               6
            3               4
3           2               4
Name: count, dtype: int64


READY FILE: /kaggle/working/submission.csv


In [8]:
# ============================================================
# V8: RECENCY-WEIGHTED RETRAINING
# Only overwrite submission.csv if local improves enough
# Current public best: V4B = 2.958113509
# ============================================================

import os
import shutil
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from catboost import CatBoostRegressor, CatBoostClassifier
from IPython.display import display

# ============================================================
# 0. BACKUP CURRENT BEST FILE
# ============================================================

if os.path.exists("/kaggle/working/submission.csv"):
    shutil.copy(
        "/kaggle/working/submission.csv",
        "/kaggle/working/submission_v4b_backup_v8.csv"
    )
    print("Backup saved: submission_v4b_backup_v8.csv")

V4B_LOCAL_SCORE = 2.3381604966141163
PUBLIC_BEST = 2.958113509

required_functions = [
    "make_training_features",
    "make_prediction_features_from_history",
    "get_feature_columns",
    "prepare_X",
    "make_score_predictions_v4",
    "evaluate_prediction",
    "apply_postprocess",
]

missing_functions = [f for f in required_functions if f not in globals()]
if missing_functions:
    raise Exception("Run V4/V4B first. Missing functions: " + str(missing_functions))

BASE_COLS = list(test.columns)
TARGET_COLS = ["team_goals", "opp_goals"]

CAT_COLS = [
    "gender",
    "team",
    "opponent",
    "tournament",
    "venue_country",
    "confederation_team",
    "confederation_opp",
]


# ============================================================
# 1. HELPERS
# ============================================================

def make_outcome_label_v8(team_goals, opp_goals):
    if team_goals > opp_goals:
        return 2
    elif team_goals == opp_goals:
        return 1
    else:
        return 0


def make_recency_weights(dates, half_life_years=20, min_weight=0.20):
    """
    Higher weight for more recent matches.
    """
    years = pd.to_datetime(dates).dt.year.astype(float)
    max_year = years.max()
    
    if half_life_years >= 900:
        return np.ones(len(years))
    
    raw = 0.5 ** ((max_year - years) / half_life_years)
    weights = min_weight + (1.0 - min_weight) * raw
    weights = weights / np.mean(weights)
    return weights.values


def cap_by_mode_v8(sub, mode="none"):
    out = sub.copy()

    def fix(tg, og):
        tg, og = int(tg), int(og)

        if mode == "none":
            return tg, og

        if tg == og:
            if mode == "strict" and tg >= 3:
                return 2, 2
            return tg, og

        if tg > og:
            if mode == "cap5":
                if tg >= 5:
                    return 4, min(1, og)
            elif mode == "cap4":
                if tg >= 5:
                    return 4, min(1, og)
                if tg == 4 and og == 0:
                    return 3, 0
            elif mode == "strict":
                if tg >= 4:
                    return 3, 0 if og == 0 else 3, 1
            return tg, og

        if og > tg:
            if mode == "cap5":
                if og >= 5:
                    return min(1, tg), 4
            elif mode == "cap4":
                if og >= 5:
                    return min(1, tg), 4
                if og == 4 and tg == 0:
                    return 0, 3
            elif mode == "strict":
                if og >= 4:
                    return 0 if tg == 0 else 1, 3
            return tg, og

        return tg, og

    pairs = out.apply(lambda r: fix(r["team_goals"], r["opp_goals"]), axis=1)
    out["team_goals"] = [p[0] for p in pairs]
    out["opp_goals"] = [p[1] for p in pairs]
    return out


# ============================================================
# 2. VALIDATION DATA
# ============================================================

cutoff_date = pd.Timestamp("2008-01-01")

hist_train = train[train["date"] < cutoff_date].copy()
val_full = train[train["date"] >= cutoff_date].copy()
val_input = val_full[BASE_COLS].copy()

print("Hist train:", hist_train.shape)
print("Validation:", val_full.shape)
print("Validation range:", val_full["date"].min(), "to", val_full["date"].max())

hist_fe = make_training_features(hist_train)
val_fe = make_prediction_features_from_history(
    val_input,
    hist_train[BASE_COLS + TARGET_COLS].copy()
)

feature_cols = get_feature_columns(hist_fe)
feature_cols = [c for c in feature_cols if c in val_fe.columns]

active_cat_cols = [c for c in CAT_COLS if c in feature_cols]

X_hist, medians = prepare_X(hist_fe, feature_cols, active_cat_cols)
X_val, _ = prepare_X(val_fe, feature_cols, active_cat_cols, medians=medians)

cat_idx = [X_hist.columns.get_loc(c) for c in active_cat_cols]

y_goal = hist_fe["team_goals"].clip(0, 10)
y_diff = (hist_fe["team_goals"] - hist_fe["opp_goals"]).clip(-8, 8)
y_total = (hist_fe["team_goals"] + hist_fe["opp_goals"]).clip(0, 12)

y_outcome = [
    make_outcome_label_v8(tg, og)
    for tg, og in zip(hist_fe["team_goals"], hist_fe["opp_goals"])
]

val_outcome_true = [
    make_outcome_label_v8(tg, og)
    for tg, og in zip(val_full["team_goals"], val_full["opp_goals"])
]


# ============================================================
# 3. V8 LOCAL TRAINING SWEEP
# ============================================================

half_life_grid = [8, 12, 18, 25, 40]

# configs around V4B / V6 / V7-success patterns
score_configs = [
    {
        "name": "v4b",
        "alpha": 0.50,
        "beta_diff": 0.32,
        "beta_total": 0.16,
        "max_goals": 5,
        "total_penalty": 0.50,
        "diff_penalty": 0.50,
        "draw_boost": 0.08,
        "oneone_boost": 0.03,
    },
    {
        "name": "v6",
        "alpha": 0.5564880990529631,
        "beta_diff": 0.3265516485119264,
        "beta_total": 0.19193477605870235,
        "max_goals": 5,
        "total_penalty": 0.5928166608611491,
        "diff_penalty": 0.5369851071102402,
        "draw_boost": 0.04,
        "oneone_boost": 0.03,
    },
    {
        "name": "outcome_plus",
        "alpha": 0.62,
        "beta_diff": 0.32,
        "beta_total": 0.16,
        "max_goals": 5,
        "total_penalty": 0.50,
        "diff_penalty": 0.50,
        "draw_boost": 0.06,
        "oneone_boost": 0.02,
    },
    {
        "name": "safe_total",
        "alpha": 0.52,
        "beta_diff": 0.34,
        "beta_total": 0.20,
        "max_goals": 5,
        "total_penalty": 0.66,
        "diff_penalty": 0.60,
        "draw_boost": 0.06,
        "oneone_boost": 0.03,
    },
]

v8_results = []
saved_models = {}

for half_life in half_life_grid:
    print("\n======================================")
    print("Training V8 half_life:", half_life)
    print("======================================")

    w = make_recency_weights(hist_fe["date"], half_life_years=half_life, min_weight=0.20)

    goal_model = CatBoostRegressor(
        iterations=700,
        learning_rate=0.035,
        depth=6,
        loss_function="RMSE",
        eval_metric="MAE",
        l2_leaf_reg=8,
        random_seed=42,
        od_type="Iter",
        od_wait=70,
        verbose=250,
    )

    goal_model.fit(
        X_hist,
        y_goal,
        cat_features=cat_idx,
        sample_weight=w,
        eval_set=(X_val, val_full["team_goals"].clip(0, 10)),
        use_best_model=True,
    )

    outcome_model = CatBoostClassifier(
        iterations=600,
        learning_rate=0.04,
        depth=6,
        loss_function="MultiClass",
        eval_metric="MultiClass",
        l2_leaf_reg=8,
        random_seed=42,
        od_type="Iter",
        od_wait=70,
        verbose=250,
    )

    outcome_model.fit(
        X_hist,
        y_outcome,
        cat_features=cat_idx,
        sample_weight=w,
        eval_set=(X_val, val_outcome_true),
        use_best_model=True,
    )

    diff_model = CatBoostRegressor(
        iterations=500,
        learning_rate=0.04,
        depth=6,
        loss_function="RMSE",
        eval_metric="MAE",
        l2_leaf_reg=8,
        random_seed=42,
        od_type="Iter",
        od_wait=70,
        verbose=250,
    )

    diff_model.fit(
        X_hist,
        y_diff,
        cat_features=cat_idx,
        sample_weight=w,
        eval_set=(X_val, (val_full["team_goals"] - val_full["opp_goals"]).clip(-8, 8)),
        use_best_model=True,
    )

    total_model = CatBoostRegressor(
        iterations=600,
        learning_rate=0.04,
        depth=6,
        loss_function="RMSE",
        eval_metric="MAE",
        l2_leaf_reg=8,
        random_seed=42,
        od_type="Iter",
        od_wait=70,
        verbose=250,
    )

    total_model.fit(
        X_hist,
        y_total,
        cat_features=cat_idx,
        sample_weight=w,
        eval_set=(X_val, (val_full["team_goals"] + val_full["opp_goals"]).clip(0, 12)),
        use_best_model=True,
    )

    val_lambda_v8 = goal_model.predict(X_val)
    val_outcome_probs_v8 = outcome_model.predict_proba(X_val)
    val_diff_pred_v8 = diff_model.predict(X_val)
    val_total_pred_v8 = total_model.predict(X_val)

    saved_models[half_life] = {
        "goal": goal_model,
        "outcome": outcome_model,
        "diff": diff_model,
        "total": total_model,
    }

    for cfg in score_configs:
        pred = make_score_predictions_v4(
            val_fe,
            val_lambda_v8,
            val_outcome_probs_v8,
            val_diff_pred_v8,
            val_total_pred_v8,
            alpha=cfg["alpha"],
            beta_diff=cfg["beta_diff"],
            beta_total=cfg["beta_total"],
            max_goals=cfg["max_goals"],
            total_penalty=cfg["total_penalty"],
            diff_penalty=cfg["diff_penalty"],
            draw_boost=cfg["draw_boost"],
            oneone_boost=cfg["oneone_boost"],
        )

        for cap_mode in ["none", "cap5", "cap4"]:
            pred_cap = cap_by_mode_v8(pred, cap_mode)
            score = evaluate_prediction(val_full, pred_cap)

            v8_results.append({
                "half_life": half_life,
                "config_name": cfg["name"],
                "cap_mode": cap_mode,
                "score": score,
                **{k: v for k, v in cfg.items() if k != "name"},
            })

v8_df = pd.DataFrame(v8_results).sort_values("score").reset_index(drop=True)

print("\nV8 TOP LOCAL SCORES:")
display(v8_df.head(30))

best_v8 = v8_df.iloc[0].to_dict()

print("\nBest V8 config:")
print(best_v8)

print("\nReference:")
print("V4B local:", V4B_LOCAL_SCORE)
print("V8 best local:", best_v8["score"])
print("Gain vs V4B:", V4B_LOCAL_SCORE - best_v8["score"])


# ============================================================
# 4. FINAL TRAINING ON ALL TRAIN ONLY IF LOCAL IMPROVED
# ============================================================

MIN_REQUIRED_GAIN = 0.015

if best_v8["score"] < V4B_LOCAL_SCORE - MIN_REQUIRED_GAIN:
    print("\nV8 improved enough. Training final model and creating submission.csv")

    best_half_life = int(best_v8["half_life"])

    all_fe = make_training_features(train)
    test_fe_v8 = make_prediction_features_from_history(
        test.copy(),
        train[BASE_COLS + TARGET_COLS].copy()
    )

    final_feature_cols = get_feature_columns(all_fe)
    final_feature_cols = [c for c in final_feature_cols if c in test_fe_v8.columns]

    final_cat_cols = [c for c in CAT_COLS if c in final_feature_cols]

    X_all, final_medians = prepare_X(all_fe, final_feature_cols, final_cat_cols)
    X_test, _ = prepare_X(test_fe_v8, final_feature_cols, final_cat_cols, medians=final_medians)

    final_cat_idx = [X_all.columns.get_loc(c) for c in final_cat_cols]

    y_all_goal = all_fe["team_goals"].clip(0, 10)
    y_all_diff = (all_fe["team_goals"] - all_fe["opp_goals"]).clip(-8, 8)
    y_all_total = (all_fe["team_goals"] + all_fe["opp_goals"]).clip(0, 12)

    y_all_outcome = [
        make_outcome_label_v8(tg, og)
        for tg, og in zip(all_fe["team_goals"], all_fe["opp_goals"])
    ]

    w_all = make_recency_weights(all_fe["date"], half_life_years=best_half_life, min_weight=0.20)

    print("Final half_life:", best_half_life)

    final_goal = CatBoostRegressor(
        iterations=700,
        learning_rate=0.035,
        depth=6,
        loss_function="RMSE",
        eval_metric="MAE",
        l2_leaf_reg=8,
        random_seed=42,
        verbose=250,
    )

    final_goal.fit(
        X_all,
        y_all_goal,
        cat_features=final_cat_idx,
        sample_weight=w_all,
    )

    final_outcome = CatBoostClassifier(
        iterations=600,
        learning_rate=0.04,
        depth=6,
        loss_function="MultiClass",
        eval_metric="MultiClass",
        l2_leaf_reg=8,
        random_seed=42,
        verbose=250,
    )

    final_outcome.fit(
        X_all,
        y_all_outcome,
        cat_features=final_cat_idx,
        sample_weight=w_all,
    )

    final_diff = CatBoostRegressor(
        iterations=500,
        learning_rate=0.04,
        depth=6,
        loss_function="RMSE",
        eval_metric="MAE",
        l2_leaf_reg=8,
        random_seed=42,
        verbose=250,
    )

    final_diff.fit(
        X_all,
        y_all_diff,
        cat_features=final_cat_idx,
        sample_weight=w_all,
    )

    final_total = CatBoostRegressor(
        iterations=600,
        learning_rate=0.04,
        depth=6,
        loss_function="RMSE",
        eval_metric="MAE",
        l2_leaf_reg=8,
        random_seed=42,
        verbose=250,
    )

    final_total.fit(
        X_all,
        y_all_total,
        cat_features=final_cat_idx,
        sample_weight=w_all,
    )

    test_lambda_v8 = final_goal.predict(X_test)
    test_outcome_probs_v8 = final_outcome.predict_proba(X_test)
    test_diff_pred_v8 = final_diff.predict(X_test)
    test_total_pred_v8 = final_total.predict(X_test)

    test_pred_v8 = make_score_predictions_v4(
        test_fe_v8,
        test_lambda_v8,
        test_outcome_probs_v8,
        test_diff_pred_v8,
        test_total_pred_v8,
        alpha=float(best_v8["alpha"]),
        beta_diff=float(best_v8["beta_diff"]),
        beta_total=float(best_v8["beta_total"]),
        max_goals=int(best_v8["max_goals"]),
        total_penalty=float(best_v8["total_penalty"]),
        diff_penalty=float(best_v8["diff_penalty"]),
        draw_boost=float(best_v8["draw_boost"]),
        oneone_boost=float(best_v8["oneone_boost"]),
    )

    test_pred_v8 = sample[["Id"]].merge(test_pred_v8, on="Id", how="left")

    test_pred_v8[["team_goals", "opp_goals"]] = (
        test_pred_v8[["team_goals", "opp_goals"]]
        .fillna(0)
        .astype(int)
    )

    test_pred_v8 = cap_by_mode_v8(test_pred_v8, best_v8["cap_mode"])

    test_pred_v8.to_csv("submission.csv", index=False)

    final_sub = test_pred_v8.copy()
    final_version = "V8"

else:
    print("\nV8 did NOT improve enough. Restoring V4B submission.csv")

    if os.path.exists("/kaggle/working/submission_v4b_backup_v8.csv"):
        shutil.copy(
            "/kaggle/working/submission_v4b_backup_v8.csv",
            "/kaggle/working/submission.csv"
        )

    final_sub = pd.read_csv("/kaggle/working/submission.csv")
    final_version = "V4B_RESTORED"


# ============================================================
# 5. FINAL CHECK
# ============================================================

print("\nFinal version in submission.csv:", final_version)
print("Saved final file: submission.csv")

print("\nSubmission check:")
print(final_sub.shape)
print(final_sub.isna().sum())
print(final_sub.head(20))

assert list(final_sub.columns) == ["Id", "team_goals", "opp_goals"]
assert final_sub.shape[0] == sample.shape[0]
assert final_sub["Id"].equals(sample["Id"])
assert final_sub[["team_goals", "opp_goals"]].isna().sum().sum() == 0

print("\nScore distribution:")
display(final_sub[["team_goals", "opp_goals"]].value_counts().head(30))

print("\nREADY FILE: /kaggle/working/submission.csv")

Backup saved: submission_v4b_backup_v8.csv
Hist train: (69842, 47)
Validation: (8930, 47)
Validation range: 2008-01-02 00:00:00 to 2011-08-04 00:00:00

Training V8 half_life: 8
0:	learn: 1.2533121	test: 1.2151245	best: 1.2151245 (0)	total: 81.2ms	remaining: 56.7s
250:	learn: 1.0039783	test: 1.0155822	best: 1.0155288 (247)	total: 15.7s	remaining: 28s
500:	learn: 0.9827969	test: 1.0122977	best: 1.0120001 (484)	total: 30.8s	remaining: 12.2s
Stopped by overfitting detector  (70 iterations wait)

bestTest = 1.012000145
bestIteration = 484

Shrink model to first 485 iterations.
0:	learn: 1.0841256	test: 1.0839336	best: 1.0839336 (0)	total: 262ms	remaining: 2m 36s
250:	learn: 0.8506666	test: 0.8859709	best: 0.8858461 (198)	total: 58s	remaining: 1m 20s
500:	learn: 0.8302826	test: 0.8842263	best: 0.8841323 (485)	total: 1m 58s	remaining: 23.4s
Stopped by overfitting detector  (70 iterations wait)

bestTest = 0.8840344649
bestIteration = 518

Shrink model to first 519 iterations.
0:	learn: 1.8263

,half_life,config_name,cap_mode,score,alpha,beta_diff,beta_total,max_goals,total_penalty,diff_penalty,draw_boost,oneone_boost
0,25,v6,none,2.325495,0.556488,0.326552,0.191935,5,0.592817,0.536985,0.04,0.03
1,25,outcome_plus,none,2.325719,0.620000,0.320000,0.160000,5,0.500000,0.500000,0.06,0.02
2,25,v6,cap5,2.326671,0.556488,0.326552,0.191935,5,0.592817,0.536985,0.04,0.03
3,25,outcome_plus,cap5,2.326708,0.620000,0.320000,0.160000,5,0.500000,0.500000,0.06,0.02
4,25,v4b,none,2.326964,0.500000,0.320000,0.160000,5,0.500000,0.500000,0.08,0.03
5,25,v4b,cap5,2.327953,0.500000,0.320000,0.160000,5,0.500000,0.500000,0.08,0.03
6,8,outcome_plus,none,2.329424,0.620000,0.320000,0.160000,5,0.500000,0.500000,0.06,0.02
7,8,outcome_plus,cap5,2.331148,0.620000,0.320000,0.160000,5,0.500000,0.500000,0.06,0.02
8,25,safe_total,cap5,2.331665,0.520000,0.340000,0.200000,5,0.660000,0.600000,0.06,0.03
9,25,safe_total,none,2.331665,0.520000,0.340000,0.200000,5,0.660000,0.600000,0.06,0.03



Best V8 config:
{'half_life': 25, 'config_name': 'v6', 'cap_mode': 'none', 'score': 2.325494723518001, 'alpha': 0.5564880990529631, 'beta_diff': 0.3265516485119264, 'beta_total': 0.19193477605870235, 'max_goals': 5, 'total_penalty': 0.5928166608611491, 'diff_penalty': 0.5369851071102402, 'draw_boost': 0.04, 'oneone_boost': 0.03}

Reference:
V4B local: 2.3381604966141163
V8 best local: 2.325494723518001
Gain vs V4B: 0.012665773096115274

V8 did NOT improve enough. Restoring V4B submission.csv

Final version in submission.csv: V4B_RESTORED
Saved final file: submission.csv

Submission check:
(42422, 3)
Id            0
team_goals    0
opp_goals     0
dtype: int64
                        Id  team_goals  opp_goals
0       M034984_Seychelles           2          1
1        M034984_Mauritius           1          2
2          M034985_Comoros           1          1
3         M034985_Maldives           1          1
4          M034986_Réunion           2          1
5       M034986_Madagascar     

team_goals  opp_goals
1           1            6464
            2            5380
2           1            5380
0           2            4151
2           0            4151
1           0            4144
0           1            4144
            3            2392
3           0            2392
0           4             852
4           0             852
1           3             757
3           1             757
0           5             257
5           0             257
1           4              39
4           1              39
2           2               6
            3               4
3           2               4
Name: count, dtype: int64


READY FILE: /kaggle/working/submission.csv


In [9]:
# ============================================================
# FORCE GENERATE V8 FINAL SUBMISSION
# Output final: submission.csv
# Best V8 config from local validation:
# half_life = 25
# config_name = v6
# cap_mode = none
# ============================================================

import os
import shutil
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from catboost import CatBoostRegressor, CatBoostClassifier
from IPython.display import display

# backup current V4B file before overwriting
if os.path.exists("/kaggle/working/submission.csv"):
    shutil.copy(
        "/kaggle/working/submission.csv",
        "/kaggle/working/submission_v4b_before_force_v8.csv"
    )
    print("Backup saved: submission_v4b_before_force_v8.csv")


# ============================================================
# REQUIRED CONFIG
# ============================================================

BEST_HALF_LIFE = 25

BEST_V8_PARAMS = {
    "alpha": 0.5564880990529631,
    "beta_diff": 0.3265516485119264,
    "beta_total": 0.19193477605870235,
    "max_goals": 5,
    "total_penalty": 0.5928166608611491,
    "diff_penalty": 0.5369851071102402,
    "draw_boost": 0.04,
    "oneone_boost": 0.03,
}

BEST_CAP_MODE = "none"

BASE_COLS = list(test.columns)
TARGET_COLS = ["team_goals", "opp_goals"]

CAT_COLS = [
    "gender",
    "team",
    "opponent",
    "tournament",
    "venue_country",
    "confederation_team",
    "confederation_opp",
]


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def make_outcome_label_v8(team_goals, opp_goals):
    if team_goals > opp_goals:
        return 2
    elif team_goals == opp_goals:
        return 1
    else:
        return 0


def make_recency_weights(dates, half_life_years=25, min_weight=0.20):
    years = pd.to_datetime(dates).dt.year.astype(float)
    max_year = years.max()
    
    raw = 0.5 ** ((max_year - years) / half_life_years)
    weights = min_weight + (1.0 - min_weight) * raw
    weights = weights / np.mean(weights)
    
    return weights.values


def cap_by_mode_v8(sub, mode="none"):
    out = sub.copy()

    def fix(tg, og):
        tg, og = int(tg), int(og)

        if mode == "none":
            return tg, og

        if tg == og:
            if mode == "strict" and tg >= 3:
                return 2, 2
            return tg, og

        if tg > og:
            if mode == "cap5":
                if tg >= 5:
                    return 4, min(1, og)
            elif mode == "cap4":
                if tg >= 5:
                    return 4, min(1, og)
                if tg == 4 and og == 0:
                    return 3, 0
            elif mode == "strict":
                if tg >= 4:
                    return 3, 0 if og == 0 else 3, 1
            return tg, og

        if og > tg:
            if mode == "cap5":
                if og >= 5:
                    return min(1, tg), 4
            elif mode == "cap4":
                if og >= 5:
                    return min(1, tg), 4
                if og == 4 and tg == 0:
                    return 0, 3
            elif mode == "strict":
                if og >= 4:
                    return 0 if tg == 0 else 1, 3
            return tg, og

        return tg, og

    pairs = out.apply(lambda r: fix(r["team_goals"], r["opp_goals"]), axis=1)
    out["team_goals"] = [p[0] for p in pairs]
    out["opp_goals"] = [p[1] for p in pairs]
    
    return out


# ============================================================
# SAFETY CHECK
# ============================================================

required_functions = [
    "make_training_features",
    "make_prediction_features_from_history",
    "get_feature_columns",
    "prepare_X",
    "make_score_predictions_v4",
]

missing_functions = [f for f in required_functions if f not in globals()]

if missing_functions:
    raise Exception("Run V4/V4B/V8 functions first. Missing: " + str(missing_functions))


# ============================================================
# FINAL TRAINING ON ALL TRAIN
# ============================================================

print("Building final V8 features...")

all_fe = make_training_features(train)

test_fe_v8 = make_prediction_features_from_history(
    test.copy(),
    train[BASE_COLS + TARGET_COLS].copy()
)

final_feature_cols = get_feature_columns(all_fe)
final_feature_cols = [c for c in final_feature_cols if c in test_fe_v8.columns]

final_cat_cols = [c for c in CAT_COLS if c in final_feature_cols]

X_all, final_medians = prepare_X(all_fe, final_feature_cols, final_cat_cols)
X_test, _ = prepare_X(test_fe_v8, final_feature_cols, final_cat_cols, medians=final_medians)

final_cat_idx = [X_all.columns.get_loc(c) for c in final_cat_cols]

y_all_goal = all_fe["team_goals"].clip(0, 10)
y_all_diff = (all_fe["team_goals"] - all_fe["opp_goals"]).clip(-8, 8)
y_all_total = (all_fe["team_goals"] + all_fe["opp_goals"]).clip(0, 12)

y_all_outcome = [
    make_outcome_label_v8(tg, og)
    for tg, og in zip(all_fe["team_goals"], all_fe["opp_goals"])
]

w_all = make_recency_weights(
    all_fe["date"],
    half_life_years=BEST_HALF_LIFE,
    min_weight=0.20
)

print("Final V8 half_life:", BEST_HALF_LIFE)
print("Training rows:", X_all.shape)
print("Test rows:", X_test.shape)


# ============================================================
# TRAIN MODELS
# ============================================================

final_goal = CatBoostRegressor(
    iterations=700,
    learning_rate=0.035,
    depth=6,
    loss_function="RMSE",
    eval_metric="MAE",
    l2_leaf_reg=8,
    random_seed=42,
    verbose=250,
)

final_goal.fit(
    X_all,
    y_all_goal,
    cat_features=final_cat_idx,
    sample_weight=w_all,
)

final_outcome = CatBoostClassifier(
    iterations=600,
    learning_rate=0.04,
    depth=6,
    loss_function="MultiClass",
    eval_metric="MultiClass",
    l2_leaf_reg=8,
    random_seed=42,
    verbose=250,
)

final_outcome.fit(
    X_all,
    y_all_outcome,
    cat_features=final_cat_idx,
    sample_weight=w_all,
)

final_diff = CatBoostRegressor(
    iterations=500,
    learning_rate=0.04,
    depth=6,
    loss_function="RMSE",
    eval_metric="MAE",
    l2_leaf_reg=8,
    random_seed=42,
    verbose=250,
)

final_diff.fit(
    X_all,
    y_all_diff,
    cat_features=final_cat_idx,
    sample_weight=w_all,
)

final_total = CatBoostRegressor(
    iterations=600,
    learning_rate=0.04,
    depth=6,
    loss_function="RMSE",
    eval_metric="MAE",
    l2_leaf_reg=8,
    random_seed=42,
    verbose=250,
)

final_total.fit(
    X_all,
    y_all_total,
    cat_features=final_cat_idx,
    sample_weight=w_all,
)


# ============================================================
# PREDICT TEST
# ============================================================

print("Predicting test...")

test_lambda_v8 = final_goal.predict(X_test)
test_outcome_probs_v8 = final_outcome.predict_proba(X_test)
test_diff_pred_v8 = final_diff.predict(X_test)
test_total_pred_v8 = final_total.predict(X_test)

test_pred_v8 = make_score_predictions_v4(
    test_fe_v8,
    test_lambda_v8,
    test_outcome_probs_v8,
    test_diff_pred_v8,
    test_total_pred_v8,
    alpha=BEST_V8_PARAMS["alpha"],
    beta_diff=BEST_V8_PARAMS["beta_diff"],
    beta_total=BEST_V8_PARAMS["beta_total"],
    max_goals=BEST_V8_PARAMS["max_goals"],
    total_penalty=BEST_V8_PARAMS["total_penalty"],
    diff_penalty=BEST_V8_PARAMS["diff_penalty"],
    draw_boost=BEST_V8_PARAMS["draw_boost"],
    oneone_boost=BEST_V8_PARAMS["oneone_boost"],
)

test_pred_v8 = sample[["Id"]].merge(test_pred_v8, on="Id", how="left")

test_pred_v8[["team_goals", "opp_goals"]] = (
    test_pred_v8[["team_goals", "opp_goals"]]
    .fillna(0)
    .astype(int)
)

test_pred_v8 = cap_by_mode_v8(test_pred_v8, BEST_CAP_MODE)

# final output
test_pred_v8.to_csv("submission.csv", index=False)


# ============================================================
# FINAL CHECK
# ============================================================

print("\nFORCE V8 READY")
print("Saved final file: submission.csv")
print("V8 params:", BEST_V8_PARAMS)
print("Cap mode:", BEST_CAP_MODE)

print("\nSubmission check:")
print(test_pred_v8.shape)
print(test_pred_v8.isna().sum())
print(test_pred_v8.head(20))

assert list(test_pred_v8.columns) == ["Id", "team_goals", "opp_goals"]
assert test_pred_v8.shape[0] == sample.shape[0]
assert test_pred_v8["Id"].equals(sample["Id"])
assert test_pred_v8[["team_goals", "opp_goals"]].isna().sum().sum() == 0

print("\nScore distribution:")
display(test_pred_v8[["team_goals", "opp_goals"]].value_counts().head(30))

print("\nREADY TO SUBMIT: /kaggle/working/submission.csv")

Backup saved: submission_v4b_before_force_v8.csv
Building final V8 features...
Final V8 half_life: 25
Training rows: (78772, 79)
Test rows: (42422, 79)
0:	learn: 1.2367884	total: 81ms	remaining: 56.6s
250:	learn: 0.9963262	total: 16.2s	remaining: 29s
500:	learn: 0.9804771	total: 31.9s	remaining: 12.7s
699:	learn: 0.9714580	total: 44.4s	remaining: 0us
0:	learn: 1.0843846	total: 265ms	remaining: 2m 38s
250:	learn: 0.8568389	total: 1m 2s	remaining: 1m 27s
500:	learn: 0.8390856	total: 2m 5s	remaining: 24.7s
599:	learn: 0.8337353	total: 2m 29s	remaining: 0us
0:	learn: 1.8015123	total: 81.1ms	remaining: 40.5s
250:	learn: 1.4247727	total: 16.1s	remaining: 16s
499:	learn: 1.3931575	total: 32s	remaining: 0us
0:	learn: 1.6000298	total: 70ms	remaining: 41.9s
250:	learn: 1.4834836	total: 15.8s	remaining: 22s
500:	learn: 1.4599595	total: 31.2s	remaining: 6.17s
599:	learn: 1.4534851	total: 37.2s	remaining: 0us
Predicting test...

FORCE V8 READY
Saved final file: submission.csv
V8 params: {'alpha': 0

team_goals  opp_goals
1           1            6324
            2            5912
2           1            5912
0           2            4094
2           0            4094
1           0            3597
0           1            3597
            3            2408
3           0            2408
0           4             997
4           0             997
1           3             739
3           1             739
0           5             261
5           0             261
1           4              33
4           1              33
2           2               8
            3               4
3           2               4
Name: count, dtype: int64


READY TO SUBMIT: /kaggle/working/submission.csv
